In [1]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text()

old = '"nzliicriminalcases": "NZLII Criminal Cases"'
new = '"nzliicriminalcases": "NZ Case Law"'

if old in text:
    text = text.replace(old, new)
    path.write_text(text)
    print("UPDATED:", path)
else:
    print("NO CHANGE: string not found")

NO CHANGE: string not found


In [2]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text(encoding="utf-8")

patterns = [
    (
        r'("nzlii_criminal_cases"\s*:\s*")NZLII Criminal Cases(")',
        r'\1NZ Case Law\2',
    ),
    (
        r'("nzliicriminalcases"\s*:\s*")NZLII Criminal Cases(")',
        r'\1NZ Case Law\2',
    ),
    (
        r"(nzlii_criminal_cases\s*[:=]\s*['\"])NZLII Criminal Cases(['\"])",
        r"\1NZ Case Law\2",
    ),
    (
        r"(nzliicriminalcases\s*[:=]\s*['\"])NZLII Criminal Cases(['\"])",
        r"\1NZ Case Law\2",
    ),
]

updated = text
replacements = 0

for pattern, repl in patterns:
    updated, count = re.subn(pattern, repl, updated)
    replacements += count

if replacements:
    path.write_text(updated, encoding="utf-8")
    print(f"UPDATED: {path}")
    print(f"Replacements made: {replacements}")
else:
    print("NO CHANGE: no matching label found")
    for m in re.finditer(r"NZLII Criminal Cases|nzlii_criminal_cases|nzliicriminalcases", text):
        start = max(0, m.start() - 120)
        end = min(len(text), m.end() + 120)
        print("\n--- CONTEXT ---\n")
        print(text[start:end])

UPDATED: /workspace/nz_legal_rag/core/rag_engine.py
Replacements made: 1


In [3]:
from pathlib import Path

PROJECT_ROOT = Path("/workspace/nz_legal_rag")
server_path = PROJECT_ROOT / "api" / "server.py"

text = server_path.read_text(encoding="utf-8")

if "def is_deletable_collection(" not in text:
    marker = 'TEMP_SESSION_PREFIX = "temp_session_"'
    insert = '''TEMP_SESSION_PREFIX = "temp_session_"
LEGACY_TEMP_PREFIX = "temp_"
PERMANENT_COLLECTIONS = {
    "nz_legislation",
    "nz_case_law",
    "nzlii_criminal_cases",
    "nz_police_manual",
}

def is_deletable_collection(name: str) -> bool:
    if not name:
        return False
    if name in PERMANENT_COLLECTIONS:
        return False
    return (
        name == "user_uploads"
        or name.startswith(TEMP_SESSION_PREFIX)
        or name.startswith(LEGACY_TEMP_PREFIX)
    )
'''
    if marker in text:
        text = text.replace(marker, insert)
        server_path.write_text(text, encoding="utf-8")
        print("UPDATED:", server_path)
    else:
        print("NO CHANGE: marker not found")
else:
    print("NO CHANGE: helper already exists")

NO CHANGE: marker not found


In [4]:
from pathlib import Path
import re

PROJECT_ROOT = Path("/workspace/nz_legal_rag")
server_path = PROJECT_ROOT / "api" / "server.py"

text = server_path.read_text(encoding="utf-8")

patterns = [
    r'for\s+name\s+in\s+.*?clear-session.*?:',
]

target = "def clear_session"
idx = text.find(target)

if idx == -1:
    print("NO CHANGE: clear_session not found")
else:
    start = idx
    next_def = text.find("\ndef ", start + 1)
    if next_def == -1:
        next_def = len(text)

    block = text[start:next_def]

    old_snippets = [
        'target_collection = f"temp_session_{session_id}"',
        'target_collection = f"temp_{session_id}"',
    ]

    new_block = block
    if 'target_collection = f"temp_session_{session_id}"' in block or 'target_collection = f"temp_{session_id}"' in block:
        new_block = re.sub(
            r'target_collection\s*=\s*f"temp_session_\{session_id\}"',
            'target_collection = f"temp_session_{session_id}"',
            new_block,
        )
        new_block = re.sub(
            r'target_collection\s*=\s*f"temp_\{session_id\}"',
            'target_collection = f"temp_{session_id}"',
            new_block,
        )

    if "is_deletable_collection" in text and new_block != block:
        text = text[:start] + new_block + text[next_def:]
        server_path.write_text(text, encoding="utf-8")
        print("UPDATED:", server_path)
    else:
        print("NO CHANGE: clear_session block not modified")
        print(block[:2000])

NO CHANGE: clear_session block not modified
def clear_session(
    session_id: str = Header(..., alias="X-Session-ID"),
    tenant=Depends(get_current_tenant),
):
    if not rag_engine:
        raise HTTPException(status_code=500, detail="RAG engine not initialized")

    if not session_id:
        raise HTTPException(status_code=400, detail="X-Session-ID header required")

    collection_name = f"temp_session_{session_id}"
    deleted = False
    files_deleted = False
    session_temp_dir = _get_session_temp_dir(session_id)

    try:
        rag_engine.client.delete_collection(name=collection_name)
        deleted = True
    except Exception:
        pass

    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]

    try:
        if session_temp_dir.exists():
            shutil.rmtree(session_temp_dir)
            files_deleted = True
    except Exception:
        pass

    return {
        "success": True,
        "deleted"

In [5]:
cd /workspace/nz_legal_rag
python3 -m py_compile api/server.py && ./restart_all.sh

SyntaxError: invalid syntax (2777221368.py, line 2)

In [6]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/workspace/nz_legal_rag")
os.chdir(PROJECT_ROOT)

result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile api/server.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 96169
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b in

In [7]:
from pathlib import Path

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

checks = [
    "LEGACY_TEMP_PREFIX",
    'TEMP_SESSION_PREFIX = "temp_session_"',
    "PERMANENT_COLLECTIONS = {",
    "def is_deletable_collection(name: str) -> bool:",
    'name == "user_uploads"',
]

for check in checks:
    print(f"{check} =>", check in text)

start = text.find("LEGACY_TEMP_PREFIX")
if start != -1:
    print("\n--- INSERTED BLOCK ---\n")
    print(text[start:start+1200])

LEGACY_TEMP_PREFIX => False
TEMP_SESSION_PREFIX = "temp_session_" => False
PERMANENT_COLLECTIONS = { => False
def is_deletable_collection(name: str) -> bool: => False
name == "user_uploads" => False


In [8]:
from pathlib import Path

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

helper_block = '''

LEGACY_TEMP_PREFIX = "temp_"
TEMP_SESSION_PREFIX = "temp_session_"
PERMANENT_COLLECTIONS = {
    "nz_legislation",
    "nz_case_law",
    "nzlii_criminal_cases",
    "nz_police_manual",
}

def is_deletable_collection(name: str) -> bool:
    if not name:
        return False
    if name in PERMANENT_COLLECTIONS:
        return False
    return (
        name == "user_uploads"
        or name.startswith(LEGACY_TEMP_PREFIX)
        or name.startswith(TEMP_SESSION_PREFIX)
    )

'''

if "def is_deletable_collection(" in text:
    print("NO CHANGE: helper already exists")
else:
    lines = text.splitlines()
    insert_idx = None

    for i, line in enumerate(lines):
        if line.startswith("@app.") or line.startswith("@router."):
            insert_idx = i
            break

    if insert_idx is None:
        print("NO CHANGE: no decorator anchor found")
    else:
        lines.insert(insert_idx, helper_block.rstrip("\n"))
        new_text = "\n".join(lines) + "\n"
        server_path.write_text(new_text, encoding="utf-8")
        print("UPDATED:", server_path)
        print("Inserted before line", insert_idx + 1)

UPDATED: /workspace/nz_legal_rag/api/server.py
Inserted before line 478


In [9]:
from pathlib import Path

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

checks = [
    "LEGACY_TEMP_PREFIX",
    'TEMP_SESSION_PREFIX = "temp_session_"',
    "PERMANENT_COLLECTIONS = {",
    "def is_deletable_collection(name: str) -> bool:",
    'name == "user_uploads"',
]

for check in checks:
    print(f"{check} =>", check in text)

start = text.find("LEGACY_TEMP_PREFIX")
if start != -1:
    print("\n--- INSERTED BLOCK ---\n")
    print(text[start:start+1200])

LEGACY_TEMP_PREFIX => True
TEMP_SESSION_PREFIX = "temp_session_" => True
PERMANENT_COLLECTIONS = { => True
def is_deletable_collection(name: str) -> bool: => True
name == "user_uploads" => True

--- INSERTED BLOCK ---

LEGACY_TEMP_PREFIX = "temp_"
TEMP_SESSION_PREFIX = "temp_session_"
PERMANENT_COLLECTIONS = {
    "nz_legislation",
    "nz_case_law",
    "nzlii_criminal_cases",
    "nz_police_manual",
}

def is_deletable_collection(name: str) -> bool:
    if not name:
        return False
    if name in PERMANENT_COLLECTIONS:
        return False
    return (
        name == "user_uploads"
        or name.startswith(LEGACY_TEMP_PREFIX)
        or name.startswith(TEMP_SESSION_PREFIX)
    )
@app.get("/")
def root():
    return {
        "name": "NZ Legal RAG API",
        "version": "1.2.2",
        "status": "operational",
        "docs": "/docs",
    }


@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "service": "nz-legal-rag-api",
        "dat

In [12]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/workspace/nz_legal_rag")
os.chdir(PROJECT_ROOT)

result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile api/server.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 98320
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b in

In [ ]:
import requests

base = "http://127.0.0.1:8000"

for endpoint in ["/health", "/health/deep"]:
    try:
        r = requests.get(base + endpoint, timeout=20)
        print(endpoint, "=>", r.status_code)
        try:
            print(r.json())
        except Exception:
            print(r.text[:1000])
        print("-" * 80)
    except Exception as e:
        print(endpoint, "=> ERROR:", e)

/health => 200
{'status': 'healthy', 'service': 'nz-legal-rag-api', 'database': {'status': 'initialized', 'path': '/workspace/chroma_db', 'deep_stats_available': True}, 'tenants': 3, 'timestamp': '2026-06-14T11:30:47.164029'}
--------------------------------------------------------------------------------


In [14]:
from pathlib import Path
import re

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

match = re.search(
    r"(def clear_session\(.*?)(\n@app\.|\n@router\.|\ndef [a-zA-Z_])",
    text,
    re.DOTALL,
)

if match:
    print(match.group(1)[:5000])
else:
    start = text.find("def clear_session")
    if start != -1:
        print(text[start:start+5000])
    else:
        print("clear_session not found")

def clear_session(
    session_id: str = Header(..., alias="X-Session-ID"),
    tenant=Depends(get_current_tenant),
):
    if not rag_engine:
        raise HTTPException(status_code=500, detail="RAG engine not initialized")

    if not session_id:
        raise HTTPException(status_code=400, detail="X-Session-ID header required")

    collection_name = f"temp_session_{session_id}"
    deleted = False
    files_deleted = False
    session_temp_dir = _get_session_temp_dir(session_id)

    try:
        rag_engine.client.delete_collection(name=collection_name)
        deleted = True
    except Exception:
        pass

    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]

    try:
        if session_temp_dir.exists():
            shutil.rmtree(session_temp_dir)
            files_deleted = True
    except Exception:
        pass

    return {
        "success": True,
        "deleted": deleted,
        "files_deleted": files_de

In [15]:
from pathlib import Path

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

checks = [
    'f"temp_session_{session_id}"',
    'f"temp_{session_id}"',
    '"user_uploads"',
    "deleted_collections",
    "failed_collections",
]

for check in checks:
    print(f"{check} =>", check in text)

start = text.find("def clear_session(")
if start != -1:
    print("\n--- clear_session ---\n")
    print(text[start:start+3000])

f"temp_session_{session_id}" => True
f"temp_{session_id}" => False
"user_uploads" => True
deleted_collections => True
failed_collections => False

--- clear_session ---

def clear_session(
    session_id: str = Header(..., alias="X-Session-ID"),
    tenant=Depends(get_current_tenant),
):
    if not rag_engine:
        raise HTTPException(status_code=500, detail="RAG engine not initialized")

    if not session_id:
        raise HTTPException(status_code=400, detail="X-Session-ID header required")

    collection_name = f"temp_session_{session_id}"
    deleted = False
    files_deleted = False
    session_temp_dir = _get_session_temp_dir(session_id)

    try:
        rag_engine.client.delete_collection(name=collection_name)
        deleted = True
    except Exception:
        pass

    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]

    try:
        if session_temp_dir.exists():
            shutil.rmtree(session_temp_dir

In [16]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/workspace/nz_legal_rag")
os.chdir(PROJECT_ROOT)

result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile api/server.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 100302
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b i

In [17]:
from pathlib import Path

server_path = Path("/workspace/nz_legal_rag/api/server.py")
text = server_path.read_text(encoding="utf-8")

checks = [
    'f"temp_session_{session_id}"',
    'f"temp_{session_id}"',
    '"user_uploads"',
    "deleted_collections",
    "failed_collections",
]

for check in checks:
    print(f"{check} =>", check in text)

start = text.find("def clear_session(")
if start != -1:
    print("\n--- clear_session ---\n")
    print(text[start:start+3000])

f"temp_session_{session_id}" => True
f"temp_{session_id}" => False
"user_uploads" => True
deleted_collections => True
failed_collections => False

--- clear_session ---

def clear_session(
    session_id: str = Header(..., alias="X-Session-ID"),
    tenant=Depends(get_current_tenant),
):
    if not rag_engine:
        raise HTTPException(status_code=500, detail="RAG engine not initialized")

    if not session_id:
        raise HTTPException(status_code=400, detail="X-Session-ID header required")

    collection_name = f"temp_session_{session_id}"
    deleted = False
    files_deleted = False
    session_temp_dir = _get_session_temp_dir(session_id)

    try:
        rag_engine.client.delete_collection(name=collection_name)
        deleted = True
    except Exception:
        pass

    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]

    try:
        if session_temp_dir.exists():
            shutil.rmtree(session_temp_dir

In [18]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

patterns = [
    r"def show_search_page\(.*?(?=\ndef |\nclass |\Z)",
    r"def search_page\(.*?(?=\ndef |\nclass |\Z)",
    r"def render_search_page\(.*?(?=\ndef |\nclass |\Z)",
    r"def show_research_page\(.*?(?=\ndef |\nclass |\Z)",
]

found = False
for pattern in patterns:
    m = re.search(pattern, text, flags=re.DOTALL)
    if m:
        print(m.group(0)[:12000])
        found = True
        break

if not found:
    for needle in [
        "Search",
        "search",
        "analysis_result",
        "legal_analysis",
        "st.markdown(",
        "sources",
        "executive_summary",
    ]:
        idx = text.find(needle)
        if idx != -1:
            start = max(0, idx - 1200)
            end = min(len(text), idx + 5000)
            print(f"\n--- around: {needle} ---\n")
            print(text[start:end])
            break


--- around: Search ---

ole"] = role
                    st.session_state["tenant_id"] = result.get("tenant_id", "")
                    st.session_state["display_name"] = result.get("name", staff_username.strip())
                    st.session_state["api_key"] = result.get("api_key", "")
                    st.session_state["quotas"] = result.get("quotas", {})
                    st.success(f"Signed in as {st.session_state['display_name']}.")
                    st.rerun()

        st.markdown("</div>", unsafe_allow_html=True)
def show_sidebar() -> str:
    with st.sidebar:
        st.markdown("## AEGIS ⚖️")
        st.caption("NZ's Legal Assistant")

        role = str(st.session_state.get("role", "client")).lower().strip() or "client"
        display_name = (
            st.session_state.get("display_name")
            or st.session_state.get("staff_username")
            or st.session_state.get("demo_email")
            or "Authenticated user"
        )

        st.info(f"Signed 

In [19]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

print("==== PAGE ROUTING CANDIDATES ====\n")
for needle in [
    'page == "🔍 Search"',
    'page == "📊 Analysis"',
    'elif page == "🔍 Search"',
    'show_search',
    'search_page',
    'render_search',
    'show_analysis',
]:
    idx = text.find(needle)
    if idx != -1:
        start = max(0, idx - 800)
        end = min(len(text), idx + 2200)
        print(f"\n--- around: {needle} ---\n")
        print(text[start:end])

print("\n==== FUNCTION CANDIDATES ====\n")
patterns = [
    r"def show_search\(.*?(?=\ndef |\Z)",
    r"def show_search_page\(.*?(?=\ndef |\Z)",
    r"def search_page\(.*?(?=\ndef |\Z)",
    r"def render_search_page\(.*?(?=\ndef |\Z)",
    r"def show_analysis\(.*?(?=\ndef |\Z)",
    r"def show_analysis_page\(.*?(?=\ndef |\Z)",
]

found_any = False
for pattern in patterns:
    for m in re.finditer(pattern, text, flags=re.DOTALL):
        print("\n--- function match ---\n")
        print(m.group(0)[:12000])
        found_any = True

if not found_any:
    print("No direct function-name match found.")

==== PAGE ROUTING CANDIDATES ====


--- around: page == "🔍 Search" ---

t.columns(3)
        m1.metric("Documents", database.get("documents", 0))
        m2.metric("Chunks", database.get("chunks", 0))
        m3.metric("Status", db_stats.get("status", "unknown"))

        st.caption(
            f"Readiness: {database.get('readiness', 'unknown')} | "
            f"Path: {database.get('path', 'unknown')}"
        )

        with st.expander("Raw database stats"):
            st.json(db_stats)
    st.markdown("</div>", unsafe_allow_html=True)


def main() -> None:
    init_session()
    if not st.session_state.get("api_key"):
        login()
        return

    page = show_sidebar()

    if page == "🏠 Home":
        show_home()
    elif page == "📤 Upload":
        show_upload()
    elif page == "🗂️ Collection Manager":
        show_collection_manager()
    elif page == "🔍 Search":
        show_search()
    elif page == "📊 Analysis":
        show_analysis()
    elif page == "👥 Admin Panel

In [20]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

new_block = '''
def render_rag_result(result: dict, title: str = "RAG Result") -> None:
    if not isinstance(result, dict):
        st.write(result)
        return

    query = result.get("query", "")
    answer = result.get("answer", "")
    executive_summary = result.get("executive_summary", "")
    confidence = result.get("confidence")
    analysis_type = result.get("analysis_type", "") or result.get("analysisType", "")
    confidence_breakdown = result.get("confidence_breakdown", "")
    audit_report = result.get("audit_report", "")
    strategic_notes = result.get("strategic_notes", "")
    sources = result.get("sources", []) or []
    citations = result.get("citations", []) or []
    agent_trace = result.get("agent_trace", {}) or {}

    st.markdown(f"### {title}")

    top_cols = st.columns([2, 1, 1])
    with top_cols[0]:
        if query:
            st.markdown(
                f'<div class="aegis-note"><strong>Query</strong><br>{query}</div>',
                unsafe_allow_html=True,
            )
    with top_cols[1]:
        if analysis_type:
            st.metric("Type", str(analysis_type).replace("_", " ").title())
    with top_cols[2]:
        if confidence is not None:
            try:
                st.metric("Confidence", f"{float(confidence):.0%}")
            except Exception:
                st.metric("Confidence", str(confidence))

    if executive_summary:
        st.markdown("### Executive Summary")
        st.markdown(
            f'<div class="aegis-panel"><div class="aegis-prose">{executive_summary}</div></div>',
            unsafe_allow_html=True,
        )

    if answer:
        st.markdown("### Legal Analysis")
        st.markdown(
            f'<div class="aegis-panel"><div class="aegis-prose">{answer}</div></div>',
            unsafe_allow_html=True,
        )

    if citations:
        st.markdown("### Citations")
        citations_html = "".join(f"<li>{c}</li>" for c in citations)
        st.markdown(
            f'<div class="aegis-list"><ul>{citations_html}</ul></div>',
            unsafe_allow_html=True,
        )

    if sources:
        st.markdown("### Retrieved Sources")
        for i, source in enumerate(sources, 1):
            metadata = source.get("metadata", {}) if isinstance(source, dict) else {}
            document = source.get("document", "") if isinstance(source, dict) else ""
            relevance = source.get("relevance", None) if isinstance(source, dict) else None
            source_name = (
                metadata.get("source")
                or metadata.get("title")
                or metadata.get("filename")
                or "Unknown source"
            )
            category = metadata.get("category", "unknown")
            page_number = metadata.get("page_number") or metadata.get("page")

            label_bits = [f"Collection: {category}"]
            if page_number:
                label_bits.append(f"Page: {page_number}")
            if relevance is not None:
                try:
                    label_bits.append(f"Relevance: {float(relevance):.1%}")
                except Exception:
                    label_bits.append(f"Relevance: {relevance}")

            with st.expander(f"{i}. {source_name}"):
                st.caption(" | ".join(label_bits))
                if document:
                    st.markdown(
                        f'<div class="aegis-prose">{document}</div>',
                        unsafe_allow_html=True,
                    )
                if metadata:
                    with st.expander("Source metadata"):
                        st.json(metadata)

    extra_sections = []
    if strategic_notes:
        extra_sections.append(("Strategic Notes", strategic_notes))
    if audit_report:
        extra_sections.append(("Audit Report", audit_report))
    if confidence_breakdown:
        extra_sections.append(("Confidence Breakdown", confidence_breakdown))

    for heading, body in extra_sections:
        with st.expander(heading):
            st.markdown(
                f'<div class="aegis-prose">{body}</div>',
                unsafe_allow_html=True,
            )

    if agent_trace:
        with st.expander("RAG Trace"):
            st.json(agent_trace)


def show_search() -> None:
    render_brand_header(show_stickman=True)
    st.markdown('<div class="aegis-panel">', unsafe_allow_html=True)
    st.markdown("## Search")
    query = st.text_area("Search query", height=120, key="search_query")

    if st.button("Run Search", use_container_width=True, key="search_btn"):
        if not query.strip():
            st.warning("Enter a search query.")
            st.markdown("</div>", unsafe_allow_html=True)
            return

        with st.spinner("Searching..."):
            result = api_call(
                "/api/v1/search",
                data={"query": query.strip()},
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        if result:
            if isinstance(result, dict):
                if "results" in result and isinstance(result["results"], list):
                    st.markdown("### Retrieved Sources")
                    for i, item in enumerate(result["results"], 1):
                        metadata = item.get("metadata", {}) if isinstance(item, dict) else {}
                        document = item.get("document", "") if isinstance(item, dict) else ""
                        relevance = item.get("relevance", None) if isinstance(item, dict) else None
                        source_name = (
                            metadata.get("source")
                            or metadata.get("title")
                            or metadata.get("filename")
                            or f"Result {i}"
                        )
                        category = metadata.get("category", "unknown")
                        page_number = metadata.get("page_number") or metadata.get("page")

                        label_bits = [f"Collection: {category}"]
                        if page_number:
                            label_bits.append(f"Page: {page_number}")
                        if relevance is not None:
                            try:
                                label_bits.append(f"Relevance: {float(relevance):.1%}")
                            except Exception:
                                label_bits.append(f"Relevance: {relevance}")

                        with st.expander(f"{i}. {source_name}"):
                            st.caption(" | ".join(label_bits))
                            if document:
                                st.markdown(
                                    f'<div class="aegis-prose">{document}</div>',
                                    unsafe_allow_html=True,
                                )
                            if metadata:
                                with st.expander("Source metadata"):
                                    st.json(metadata)
                else:
                    render_rag_result(result, title="Search Result")
            elif isinstance(result, list):
                st.markdown("### Retrieved Sources")
                for i, item in enumerate(result, 1):
                    if isinstance(item, dict):
                        metadata = item.get("metadata", {})
                        document = item.get("document", "")
                        source_name = (
                            metadata.get("source")
                            or metadata.get("title")
                            or metadata.get("filename")
                            or f"Result {i}"
                        )
                        with st.expander(f"{i}. {source_name}"):
                            st.caption(f"Collection: {metadata.get('category', 'unknown')}")
                            if document:
                                st.markdown(
                                    f'<div class="aegis-prose">{document}</div>',
                                    unsafe_allow_html=True,
                                )
                            with st.expander("Source metadata"):
                                st.json(metadata)
                    else:
                        st.write(item)
            else:
                st.write(result)

    st.markdown("</div>", unsafe_allow_html=True)


def show_analysis() -> None:
    render_brand_header(show_stickman=True)
    st.markdown('<div class="aegis-panel">', unsafe_allow_html=True)
    st.markdown("## Analysis")

    query = st.text_area("Analysis request", height=160, key="analysis_query")
    analysis_type = st.selectbox(
        "Analysis type",
        ["general", "charge_review", "similar_cases", "element_check"],
        index=0,
        key="analysis_type",
    )

    if st.button("Analyze", type="primary", use_container_width=True, key="analyze_btn"):
        if not query.strip():
            st.warning("Enter an analysis request.")
            st.markdown("</div>", unsafe_allow_html=True)
            return

        with st.spinner("Analysing..."):
            result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query.strip(),
                    "analysis_type": analysis_type,
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        if result:
            if isinstance(result, dict):
                render_rag_result(result, title="Analysis Result")
            else:
                st.write(result)

    uploaded_docs = st.session_state.get("uploaded_documents", [])
    if uploaded_docs:
        files_html = "".join(f"<li>{name}</li>" for name in uploaded_docs)
        st.markdown(
            f'<div class="aegis-list"><strong>Recently uploaded files</strong><ul>{files_html}</ul></div>',
            unsafe_allow_html=True,
        )

    st.markdown("</div>", unsafe_allow_html=True)
'''

pattern = r'def show_search\(\) -> None:.*?def show_admin_panel\(\) -> None:'
replacement = new_block + '\n\ndef show_admin_panel() -> None:'
new_text, count = re.subn(pattern, replacement, text, flags=re.DOTALL)

if count != 1:
    raise RuntimeError(f"Expected to replace 1 search/analysis block, replaced {count}")

path.write_text(new_text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [21]:
import os
import subprocess

os.chdir("/workspace/nz_legal_rag")
result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile web/streamlit_app.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 102475
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b i

In [22]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

pattern = r'''
def show_search\(\) -> None:
    render_brand_header\(show_stickman=True\)
    st\.markdown\('<div class="aegis-panel">', unsafe_allow_html=True\)
    st\.markdown\("## Search"\)
    query = st\.text_area\("Search query", height=120, key="search_query"\)

    if st\.button\("Run Search", use_container_width=True, key="search_btn"\):
        if not query\.strip\(\):
            st\.warning\("Enter a search query\."\)
            st\.markdown\("</div>", unsafe_allow_html=True\)
            return

        with st\.spinner\("Searching\.\.\."\):
            result = api_call\(
                "/api/v1/search",
                data=\{"query": query\.strip\(\)\},
                method="POST",
                api_key=st\.session_state\.get\("api_key"\),
                session_id=st\.session_state\.get\("session_id"\),
            \)

        if result:
            if isinstance\(result, dict\):
                if "results" in result and isinstance\(result\["results"\], list\):
                    st\.markdown\("### Retrieved Sources"\)
                    for i, item in enumerate\(result\["results"\], 1\):
                        metadata = item.get\("metadata", {}\) if isinstance\(item, dict\) else {}
                        document = item.get\("document", ""\) if isinstance\(item, dict\) else ""
                        relevance = item.get\("relevance", None\) if isinstance\(item, dict\) else None
                        source_name = \(
                            metadata.get\("source"\)
                            or metadata.get\("title"\)
                            or metadata.get\("filename"\)
                            or f"Result \{i\}"
                        \)
                        category = metadata.get\("category", "unknown"\)
                        page_number = metadata.get\("page_number"\) or metadata.get\("page"\)

                        label_bits = \[f"Collection: \{category\}"\]
                        if page_number:
                            label_bits.append\(f"Page: \{page_number\}"\)
                        if relevance is not None:
                            try:
                                label_bits.append\(f"Relevance: \{float\(relevance\):.1%\}"\)
                            except Exception:
                                label_bits.append\(f"Relevance: \{relevance\}"\)

                        with st\.expander\(f"\{i\}. \{source_name\}"\):
                            st\.caption\(" | ".join\(label_bits\)\)
                            if document:
                                st\.markdown\(
                                    f'<div class="aegis-prose">\{document\}</div>',
                                    unsafe_allow_html=True,
                                \)
                            if metadata:
                                with st\.expander\("Source metadata"\):
                                    st\.json\(metadata\)
                else:
                    render_rag_result\(result, title="Search Result"\)
            elif isinstance\(result, list\):
                st\.markdown\("### Retrieved Sources"\)
                for i, item in enumerate\(result, 1\):
                    if isinstance\(item, dict\):
                        metadata = item.get\("metadata", {}\)
                        document = item.get\("document", ""\)
                        source_name = \(
                            metadata.get\("source"\)
                            or metadata.get\("title"\)
                            or metadata.get\("filename"\)
                            or f"Result \{i\}"
                        \)
                        with st\.expander\(f"\{i\}. \{source_name\}"\):
                            st\.caption\(f"Collection: \{metadata.get\('category', 'unknown'\)}"\)
                            if document:
                                st\.markdown\(
                                    f'<div class="aegis-prose">\{document\}</div>',
                                    unsafe_allow_html=True,
                                \)
                            with st\.expander\("Source metadata"\):
                                st\.json\(metadata\)
                    else:
                        st\.write\(item\)
            else:
                st\.write\(result\)

    st\.markdown\("</div>", unsafe_allow_html=True\)
'''

In [23]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

new_func = '''
def show_search() -> None:
    render_brand_header(show_stickman=True)
    st.markdown('<div class="aegis-panel">', unsafe_allow_html=True)
    st.markdown("## Search")
    st.markdown(
        '<div class="aegis-note">Ask a legal question in plain English. AEGIS will explain first, then show the retrieved sources underneath.</div>',
        unsafe_allow_html=True,
    )

    query = st.text_area("Search query", height=120, key="search_query")

    if st.button("Run Search", use_container_width=True, key="search_btn"):
        if not query.strip():
            st.warning("Enter a search query.")
            st.markdown("</div>", unsafe_allow_html=True)
            return

        with st.spinner("Researching and explaining..."):
            result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query.strip(),
                    "analysis_type": "general",
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        if result:
            if not isinstance(result, dict):
                st.write(result)
                st.markdown("</div>", unsafe_allow_html=True)
                return

            query_text = result.get("query", query.strip())
            answer = result.get("answer", "")
            executive_summary = result.get("executive_summary", "")
            confidence = result.get("confidence")
            sources = result.get("sources", []) or []
            citations = result.get("citations", []) or []

            header_cols = st.columns([3, 1])
            with header_cols[0]:
                st.markdown(
                    f'<div class="aegis-note"><strong>Question</strong><br>{query_text}</div>',
                    unsafe_allow_html=True,
                )
            with header_cols[1]:
                if confidence is not None:
                    try:
                        st.metric("Confidence", f"{float(confidence):.0%}")
                    except Exception:
                        st.metric("Confidence", str(confidence))

            if executive_summary:
                st.markdown("### Explanation")
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{executive_summary}</div></div>',
                    unsafe_allow_html=True,
                )

            if answer:
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{answer}</div></div>',
                    unsafe_allow_html=True,
                )

            if citations:
                with st.expander("View citations"):
                    citations_html = "".join(f"<li>{c}</li>" for c in citations)
                    st.markdown(
                        f'<div class="aegis-list"><ul>{citations_html}</ul></div>',
                        unsafe_allow_html=True,
                    )

            with st.expander(f"View retrieved sources ({len(sources)})", expanded=False):
                if not sources:
                    st.caption("No source list returned.")
                else:
                    for i, source in enumerate(sources, 1):
                        metadata = source.get("metadata", {}) if isinstance(source, dict) else {}
                        document = source.get("document", "") if isinstance(source, dict) else ""
                        relevance = source.get("relevance", None) if isinstance(source, dict) else None

                        source_name = (
                            metadata.get("source")
                            or metadata.get("title")
                            or metadata.get("filename")
                            or f"Source {i}"
                        )
                        category = metadata.get("category", "unknown")
                        page_number = metadata.get("page_number") or metadata.get("page")

                        bits = [f"Collection: {category}"]
                        if page_number:
                            bits.append(f"Page: {page_number}")
                        if relevance is not None:
                            try:
                                bits.append(f"Relevance: {float(relevance):.1%}")
                            except Exception:
                                bits.append(f"Relevance: {relevance}")

                        with st.expander(f"{i}. {source_name}", expanded=False):
                            st.caption(" | ".join(bits))
                            if document:
                                st.markdown(
                                    f'<div class="aegis-prose">{document}</div>',
                                    unsafe_allow_html=True,
                                )
                            if metadata:
                                with st.expander("Source metadata", expanded=False):
                                    st.json(metadata)

    st.markdown("</div>", unsafe_allow_html=True)
'''

pattern = r'def show_search\(\) -> None:.*?^\s*def show_analysis\(\) -> None:'
replacement = new_func + '\n\ndef show_analysis() -> None:'
new_text, count = re.subn(pattern, replacement, text, flags=re.DOTALL | re.MULTILINE)

if count != 1:
    raise RuntimeError(f"Expected to replace 1 show_search() block, replaced {count}")

path.write_text(new_text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [24]:
import os
import subprocess

os.chdir("/workspace/nz_legal_rag")
result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile web/streamlit_app.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 105065
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b i

In [25]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

new_func = '''
def show_search() -> None:
    render_brand_header(show_stickman=True)
    st.markdown('<div class="aegis-panel">', unsafe_allow_html=True)
    st.markdown("## Search")
    st.markdown(
        '<div class="aegis-note">Ask a legal question in plain English. AEGIS will explain first, then show the verified retrieved sources underneath.</div>',
        unsafe_allow_html=True,
    )

    query = st.text_area("Search query", height=120, key="search_query")

    if st.button("Run Search", use_container_width=True, key="search_btn"):
        if not query.strip():
            st.warning("Enter a search query.")
            st.markdown("</div>", unsafe_allow_html=True)
            return

        query_text = query.strip()

        with st.spinner("Researching and explaining..."):
            analysis_result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query_text,
                    "analysis_type": "general",
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        with st.spinner("Loading verified source list..."):
            search_result = api_call(
                "/api/v1/search",
                data={"query": query_text},
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        if analysis_result and isinstance(analysis_result, dict):
            display_query = analysis_result.get("query", query_text)
            answer = analysis_result.get("answer", "")
            executive_summary = analysis_result.get("executive_summary", "")
            confidence = analysis_result.get("confidence")
            citations = analysis_result.get("citations", []) or []

            header_cols = st.columns([3, 1])
            with header_cols[0]:
                st.markdown(
                    f'<div class="aegis-note"><strong>Question</strong><br>{display_query}</div>',
                    unsafe_allow_html=True,
                )
            with header_cols[1]:
                if confidence is not None:
                    try:
                        st.metric("Confidence", f"{float(confidence):.0%}")
                    except Exception:
                        st.metric("Confidence", str(confidence))

            st.markdown("### Explanation")
            if executive_summary:
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{executive_summary}</div></div>',
                    unsafe_allow_html=True,
                )
            if answer:
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{answer}</div></div>',
                    unsafe_allow_html=True,
                )

            if citations:
                with st.expander("View citations", expanded=False):
                    citations_html = "".join(f"<li>{c}</li>" for c in citations)
                    st.markdown(
                        f'<div class="aegis-list"><ul>{citations_html}</ul></div>',
                        unsafe_allow_html=True,
                    )
        elif analysis_result:
            st.write(analysis_result)

        verified_sources = []
        if isinstance(search_result, dict):
            if isinstance(search_result.get("results"), list):
                verified_sources = search_result.get("results", [])
            elif isinstance(search_result.get("sources"), list):
                verified_sources = search_result.get("sources", [])
        elif isinstance(search_result, list):
            verified_sources = search_result

        with st.expander(f"View retrieved sources ({len(verified_sources)})", expanded=False):
            if not verified_sources:
                st.caption("No verified sources returned from /api/v1/search.")
            else:
                for i, item in enumerate(verified_sources, 1):
                    if not isinstance(item, dict):
                        with st.expander(f"{i}. Source {i}", expanded=False):
                            st.write(item)
                        continue

                    metadata = item.get("metadata", {}) or {}
                    document = item.get("document", "") or item.get("text", "") or item.get("content", "")
                    relevance = item.get("relevance", None)

                    source_name = (
                        metadata.get("source")
                        or metadata.get("title")
                        or metadata.get("filename")
                        or item.get("source")
                        or item.get("title")
                        or f"Source {i}"
                    )
                    category = (
                        metadata.get("category")
                        or item.get("category")
                        or "unknown"
                    )
                    page_number = (
                        metadata.get("page_number")
                        or metadata.get("page")
                        or item.get("page_number")
                        or item.get("page")
                    )

                    bits = [f"Collection: {category}"]
                    if page_number:
                        bits.append(f"Page: {page_number}")
                    if relevance is not None:
                        try:
                            bits.append(f"Relevance: {float(relevance):.1%}")
                        except Exception:
                            bits.append(f"Relevance: {relevance}")

                    with st.expander(f"{i}. {source_name}", expanded=False):
                        st.caption(" | ".join(bits))
                        if document:
                            st.markdown(
                                f'<div class="aegis-prose">{document}</div>',
                                unsafe_allow_html=True,
                            )
                        else:
                            st.caption("No preview text returned for this source.")
                        merged_meta = {}
                        if isinstance(metadata, dict):
                            merged_meta.update(metadata)
                        for k in ("source", "title", "category", "page", "page_number", "relevance"):
                            if k in item and k not in merged_meta:
                                merged_meta[k] = item[k]
                        if merged_meta:
                            with st.expander("Source metadata", expanded=False):
                                st.json(merged_meta)

    st.markdown("</div>", unsafe_allow_html=True)
'''

pattern = r'def show_search\(\) -> None:.*?^\s*def show_analysis\(\) -> None:'
replacement = new_func + '\n\ndef show_analysis() -> None:'
new_text, count = re.subn(pattern, replacement, text, flags=re.DOTALL | re.MULTILINE)

if count != 1:
    raise RuntimeError(f"Expected to replace 1 show_search() block, replaced {count}")

path.write_text(new_text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [26]:
import os
import subprocess

os.chdir("/workspace/nz_legal_rag")
result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile web/streamlit_app.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 107618
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b i

In [27]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

new_func = '''
def show_search() -> None:
    render_brand_header(show_stickman=True)
    st.markdown('<div class="aegis-panel">', unsafe_allow_html=True)
    st.markdown("## Search")
    st.markdown(
        '<div class="aegis-note">Ask a legal question in plain English. AEGIS will explain first, then show the verified retrieved sources underneath.</div>',
        unsafe_allow_html=True,
    )

    query = st.text_area("Search query", height=120, key="search_query")

    if st.button("Run Search", use_container_width=True, key="search_btn"):
        if not query.strip():
            st.warning("Enter a search query.")
            st.markdown("</div>", unsafe_allow_html=True)
            return

        query_text = query.strip()

        with st.spinner("Researching and explaining..."):
            analysis_result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query_text,
                    "analysis_type": "general",
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        with st.spinner("Loading verified source list..."):
            search_result = api_call(
                "/api/v1/search",
                data={"query": query_text},
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        if analysis_result and isinstance(analysis_result, dict):
            display_query = analysis_result.get("query", query_text)
            answer = analysis_result.get("answer", "")
            executive_summary = analysis_result.get("executive_summary", "")
            confidence = analysis_result.get("confidence")
            citations = analysis_result.get("citations", []) or []

            header_cols = st.columns([3, 1])
            with header_cols[0]:
                st.markdown(
                    f'<div class="aegis-note"><strong>Question</strong><br>{display_query}</div>',
                    unsafe_allow_html=True,
                )
            with header_cols[1]:
                if confidence is not None:
                    try:
                        st.metric("Confidence", f"{float(confidence):.0%}")
                    except Exception:
                        st.metric("Confidence", str(confidence))

            st.markdown("### Explanation")
            if executive_summary:
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{executive_summary}</div></div>',
                    unsafe_allow_html=True,
                )
            if answer:
                st.markdown(
                    f'<div class="aegis-panel"><div class="aegis-prose">{answer}</div></div>',
                    unsafe_allow_html=True,
                )

            if citations:
                with st.expander("View citations", expanded=False):
                    citations_html = "".join(f"<li>{c}</li>" for c in citations)
                    st.markdown(
                        f'<div class="aegis-list"><ul>{citations_html}</ul></div>',
                        unsafe_allow_html=True,
                    )
        elif analysis_result:
            st.write(analysis_result)

        verified_sources = []
        if isinstance(search_result, dict):
            if isinstance(search_result.get("results"), list):
                verified_sources = search_result.get("results", [])
            elif isinstance(search_result.get("sources"), list):
                verified_sources = search_result.get("sources", [])
        elif isinstance(search_result, list):
            verified_sources = search_result

        with st.expander(f"View retrieved sources ({len(verified_sources)})", expanded=False):
            if not verified_sources:
                st.caption("No verified sources returned from /api/v1/search.")
            else:
                for i, item in enumerate(verified_sources, 1):
                    if not isinstance(item, dict):
                        with st.expander(f"{i}. Source {i}", expanded=False):
                            st.write(item)
                        continue

                    metadata = item.get("metadata", {}) or {}
                    document = item.get("document", "") or item.get("text", "") or item.get("content", "")
                    relevance = item.get("relevance", None)

                    source_name = (
                        metadata.get("source")
                        or metadata.get("title")
                        or metadata.get("filename")
                        or item.get("source")
                        or item.get("title")
                        or f"Source {i}"
                    )
                    category = (
                        metadata.get("category")
                        or item.get("category")
                        or "unknown"
                    )
                    page_number = (
                        metadata.get("page_number")
                        or metadata.get("page")
                        or item.get("page_number")
                        or item.get("page")
                    )

                    bits = [f"Collection: {category}"]
                    if page_number:
                        bits.append(f"Page: {page_number}")
                    if relevance is not None:
                        try:
                            bits.append(f"Relevance: {float(relevance):.1%}")
                        except Exception:
                            bits.append(f"Relevance: {relevance}")

                    with st.expander(f"{i}. {source_name}", expanded=False):
                        st.caption(" | ".join(bits))
                        if document:
                            st.markdown(
                                f'<div class="aegis-prose">{document}</div>',
                                unsafe_allow_html=True,
                            )
                        else:
                            st.caption("No preview text returned for this source.")
                        merged_meta = {}
                        if isinstance(metadata, dict):
                            merged_meta.update(metadata)
                        for k in ("source", "title", "category", "page", "page_number", "relevance"):
                            if k in item and k not in merged_meta:
                                merged_meta[k] = item[k]
                        if merged_meta:
                            with st.expander("Source metadata", expanded=False):
                                st.json(merged_meta)

    st.markdown("</div>", unsafe_allow_html=True)
'''

pattern = r'def show_search\(\) -> None:.*?^\s*def show_analysis\(\) -> None:'
replacement = new_func + '\n\ndef show_analysis() -> None:'
new_text, count = re.subn(pattern, replacement, text, flags=re.DOTALL | re.MULTILINE)

if count != 1:
    raise RuntimeError(f"Expected to replace 1 show_search() block, replaced {count}")

path.write_text(new_text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [28]:
import os
import subprocess

os.chdir("/workspace/nz_legal_rag")
result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile web/streamlit_app.py && ./restart_all.sh"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout[-12000:])
print("\n--- STDERR ---\n")
print(result.stderr[-12000:])

RETURN CODE: 0

--- STDOUT ---

═══════════════════════════════════════════════════════════════
  NZ Legal RAG - Full Shutdown & Restart
═══════════════════════════════════════════════════════════════

This script will:
  1. Stop all running services (API, Streamlit, Ollama)
  2. Clear ports and resources
  3. Verify/install Ollama and models
  4. Start all services fresh

[PHASE 1/4] Stopping all services...

Stopping Streamlit...
  ✓ Streamlit stopped
Stopping API server...
  ✓ API server stopped
Stopping Ollama...
  ✓ Ollama stopped
Cleaning up remaining processes...
Force clearing ports...

✓ All services stopped

[PHASE 2/4] Verifying clean state...

  ✓ Port 8000 is clear
  ✓ Port 8501 is clear
  ✓ Port 11434 is clear

✓ System is clean

[PHASE 3/4] Setting up Ollama...

  ✓ Ollama already installed
Starting Ollama server...
  Ollama PID: 329705
Waiting for Ollama server...
.  ✓ Ollama server ready

Checking required models...
  ✓ all-minilm:latest installed
  ✓ deepseek-r1:14b i

In [29]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
    'def api_call(\n    endpoint: str,\n    data: dict | None = None,\n    method: str = "POST",\n    api_key: str | None = None,\n    session_id: str | None = None,\n    files: list | None = None,\n    file_field_name: str = "files",\n    quiet: bool = False,\n) -> Any:\n',
    'def api_call(\n    endpoint: str,\n    data: dict | None = None,\n    method: str = "POST",\n    api_key: str | None = None,\n    session_id: str | None = None,\n    files: list | None = None,\n    file_field_name: str = "files",\n    quiet: bool = False,\n    connect_timeout: int = 15,\n    read_timeout: int = 60,\n) -> Any:\n'
)

text = text.replace(
    '                timeout=(30, 600),',
    '                timeout=(connect_timeout, read_timeout),'
)

text = text.replace(
    '                resp = requests.get(url, headers=headers, params=data, timeout=(30, 600))',
    '                resp = requests.get(url, headers=headers, params=data, timeout=(connect_timeout, read_timeout))'
)

text = text.replace(
    '                resp = requests.request(\n                    method.upper(),\n                    url,\n                    headers=headers,\n                    json=data,\n                    timeout=(30, 600),\n                )',
    '                resp = requests.request(\n                    method.upper(),\n                    url,\n                    headers=headers,\n                    json=data,\n                    timeout=(connect_timeout, read_timeout),\n                )'
)

text = text.replace(
    '        if destination == "Temporary session":\n            target_collection = f"session_{st.session_state.get(\'session_id\', \'default\')}"',
    '        if destination == "Temporary session":\n            target_collection = f"temp_session_{st.session_state.get(\'session_id\', \'default\')}"'
)

text = text.replace(
    '        with st.spinner("Searching..."):\n            result = api_call(\n                "/api/v1/search",\n                data={"query": query.strip()},\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n            )',
    '        with st.spinner("Searching..."):\n            result = api_call(\n                "/api/v1/search",\n                data={"query": query.strip()},\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n                connect_timeout=10,\n                read_timeout=45,\n            )'
)

text = text.replace(
    '        with st.spinner("Analysing..."):\n            result = api_call(\n                "/api/v1/analyze",\n                data={\n                    "query": query.strip(),\n                    "analysis_type": analysis_type,\n                },\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n            )',
    '        with st.spinner("Analysing..."):\n            result = api_call(\n                "/api/v1/analyze",\n                data={\n                    "query": query.strip(),\n                    "analysis_type": analysis_type,\n                },\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n                connect_timeout=15,\n                read_timeout=180,\n            )'
)

text = text.replace(
    '    if collection.startswith("temp_"):\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required for temporary uploads")\n        target_collection = f"temp_session_{session_id}"',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    if is_temp_upload:\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required for temporary uploads")\n        target_collection = f"temp_session_{session_id}"'
)

text = text.replace(
    '    check_quota(tenant, "store_permanent" if not collection.startswith("temp_") else "query")',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    check_quota(tenant, "query" if is_temp_upload else "store_permanent")'
)

text = text.replace(
    '    if collection.startswith("temp_"):\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required")\n        target_collection = f"temp_session_{session_id}"',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    if is_temp_upload:\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required")\n        target_collection = f"temp_session_{session_id}"'
)

text = text.replace(
    '        result = purge_stale_temp_session_dirs()',
    '        result = purge_stale_temp_sessions()'
)

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [30]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
    'def api_call(\n    endpoint: str,\n    data: dict | None = None,\n    method: str = "POST",\n    api_key: str | None = None,\n    session_id: str | None = None,\n    files: list | None = None,\n    file_field_name: str = "files",\n    quiet: bool = False,\n) -> Any:\n',
    'def api_call(\n    endpoint: str,\n    data: dict | None = None,\n    method: str = "POST",\n    api_key: str | None = None,\n    session_id: str | None = None,\n    files: list | None = None,\n    file_field_name: str = "files",\n    quiet: bool = False,\n    connect_timeout: int = 15,\n    read_timeout: int = 60,\n) -> Any:\n'
)

text = text.replace(
    '                timeout=(30, 600),',
    '                timeout=(connect_timeout, read_timeout),'
)

text = text.replace(
    '                resp = requests.get(url, headers=headers, params=data, timeout=(30, 600))',
    '                resp = requests.get(url, headers=headers, params=data, timeout=(connect_timeout, read_timeout))'
)

text = text.replace(
    '                resp = requests.request(\n                    method.upper(),\n                    url,\n                    headers=headers,\n                    json=data,\n                    timeout=(30, 600),\n                )',
    '                resp = requests.request(\n                    method.upper(),\n                    url,\n                    headers=headers,\n                    json=data,\n                    timeout=(connect_timeout, read_timeout),\n                )'
)

text = text.replace(
    '        if destination == "Temporary session":\n            target_collection = f"session_{st.session_state.get(\'session_id\', \'default\')}"',
    '        if destination == "Temporary session":\n            target_collection = f"temp_session_{st.session_state.get(\'session_id\', \'default\')}"'
)

text = text.replace(
    '        with st.spinner("Searching..."):\n            result = api_call(\n                "/api/v1/search",\n                data={"query": query.strip()},\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n            )',
    '        with st.spinner("Searching..."):\n            result = api_call(\n                "/api/v1/search",\n                data={"query": query.strip()},\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n                connect_timeout=10,\n                read_timeout=45,\n            )'
)

text = text.replace(
    '        with st.spinner("Analysing..."):\n            result = api_call(\n                "/api/v1/analyze",\n                data={\n                    "query": query.strip(),\n                    "analysis_type": analysis_type,\n                },\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n            )',
    '        with st.spinner("Analysing..."):\n            result = api_call(\n                "/api/v1/analyze",\n                data={\n                    "query": query.strip(),\n                    "analysis_type": analysis_type,\n                },\n                method="POST",\n                api_key=st.session_state.get("api_key"),\n                session_id=st.session_state.get("session_id"),\n                connect_timeout=15,\n                read_timeout=180,\n            )'
)

text = text.replace(
    '    if collection.startswith("temp_"):\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required for temporary uploads")\n        target_collection = f"temp_session_{session_id}"',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    if is_temp_upload:\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required for temporary uploads")\n        target_collection = f"temp_session_{session_id}"'
)

text = text.replace(
    '    check_quota(tenant, "store_permanent" if not collection.startswith("temp_") else "query")',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    check_quota(tenant, "query" if is_temp_upload else "store_permanent")'
)

text = text.replace(
    '    if collection.startswith("temp_"):\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required")\n        target_collection = f"temp_session_{session_id}"',
    '    is_temp_upload = collection.startswith("temp_") or collection.startswith("session_")\n    if is_temp_upload:\n        if not session_id:\n            raise HTTPException(status_code=400, detail="X-Session-ID header required")\n        target_collection = f"temp_session_{session_id}"'
)

text = text.replace(
    '        result = purge_stale_temp_session_dirs()',
    '        result = purge_stale_temp_sessions()'
)

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py


In [31]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text(encoding="utf-8")

old_ingest_document = '''    def ingest_document(self, 
                        file_path: str,
                        collection: str = "confidential",
                        metadata: Optional[Dict] = None) -> str:
        """
        Ingest a new document into the database
        """
        if collection not in self.collections:
            # Create collection if it doesn't exist
            self.collections[collection] = self.client.create_collection(
                name=collection,
                metadata={"description": f"Collection for {collection}"}
            )
'''

new_ingest_document = '''    def ingest_document(self, 
                        file_path: str,
                        collection: str = "confidential",
                        metadata: Optional[Dict] = None) -> str:
        """
        Ingest a new document into the database
        """
        if collection not in self.collections:
            try:
                self.collections[collection] = self.client.get_collection(collection)
            except Exception:
                self.collections[collection] = self.client.create_collection(
                    name=collection,
                    metadata={"description": f"Collection for {collection}"}
                )
'''

old_ingest_text = '''    def ingest_text(self,
                    documents: List[Dict[str, Any]],
                    collection: str = "user_uploads",
                    metadata: Optional[Dict] = None) -> str:
        """
        Ingest text documents into a collection.
        Supports optional page-aware metadata via document keys:
        - name
        - content
        - file_type
        - page_texts: [{"page_number": int, "text": str}]
        """
        if collection not in self.collections:
            self.collections[collection] = self.client.create_collection(
                name=collection,
                metadata={"description": f"Collection for {collection}"}
            )
'''

new_ingest_text = '''    def ingest_text(self,
                    documents: List[Dict[str, Any]],
                    collection: str = "user_uploads",
                    metadata: Optional[Dict] = None) -> str:
        """
        Ingest text documents into a collection.
        Supports optional page-aware metadata via document keys:
        - name
        - content
        - file_type
        - page_texts: [{"page_number": int, "text": str}]
        """
        if collection not in self.collections:
            try:
                self.collections[collection] = self.client.get_collection(collection)
            except Exception:
                self.collections[collection] = self.client.create_collection(
                    name=collection,
                    metadata={"description": f"Collection for {collection}"}
                )
'''

text = text.replace(old_ingest_document, new_ingest_document)
text = text.replace(old_ingest_text, new_ingest_text)

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/core/rag_engine.py


In [32]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
'''        if collection not in self.collections:
            # Create collection if it doesn't exist
            self.collections[collection] = self.client.create_collection(
                name=collection,
                metadata={"description": f"Collection for {collection}"}
            )
''',
'''        if collection not in self.collections:
            try:
                self.collections[collection] = self.client.get_collection(collection)
            except Exception:
                self.collections[collection] = self.client.create_collection(
                    name=collection,
                    metadata={"description": f"Collection for {collection}"}
                )
'''
)

text = text.replace(
'''        if collection not in self.collections:
            self.collections[collection] = self.client.create_collection(
                name=collection,
                metadata={"description": f"Collection for {collection}"}
            )
''',
'''        if collection not in self.collections:
            try:
                self.collections[collection] = self.client.get_collection(collection)
            except Exception:
                self.collections[collection] = self.client.create_collection(
                    name=collection,
                    metadata={"description": f"Collection for {collection}"}
                )
'''
)

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/core/rag_engine.py


In [33]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
    '_log("STEP 1: creating NZLegalRAG")',
    '_log("STEP 1: creating NZLegalRAG")\n        _log(f"Embedding model: {os.getenv(\'EMBEDDING_MODEL\', \'nomic-embed-text:latest\')}")'
)

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/api/server.py


In [34]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
    '_log("STEP 1: creating NZLegalRAG")',
    '_log("STEP 1: creating NZLegalRAG")\n        _log(f"Embedding model: {os.getenv(\'EMBEDDING_MODEL\', \'nomic-embed-text:latest\')}")'
)

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/api/server.py


In [35]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

old = '''        collection = st.selectbox(
            "Collection",
            collection_options,
            index=0,
            help="Choose where uploaded documents will be stored",
        )
'''

new = '''        collection = st.selectbox(
            "Collection",
            collection_options,
            index=0,
            help="Choose where uploaded documents will be stored",
        )

        new_collection_name = st.text_input(
            "Or create/use a new collection",
            value="user_uploads_384",
            help="Use a fresh collection name when an older collection was built with a different embedding model.",
        )

        if new_collection_name.strip():
            collection = new_collection_name.strip()
'''

text = text.replace(old, new)

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/web/streamlit_app.py


In [36]:
from pathlib import Path
import os

env_path = Path("/workspace/nz_legal_rag/.env")
text = env_path.read_text(encoding="utf-8") if env_path.exists() else ""

lines = [line for line in text.splitlines() if not line.startswith("EMBEDDING_MODEL=")]
lines.append("EMBEDDING_MODEL=nomic-embed-text:latest")
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"Updated {env_path}")

Updated /workspace/nz_legal_rag/.env


In [37]:
from pathlib import Path
import chromadb
from langchain_ollama import OllamaEmbeddings

DB_PATH = "/workspace/chroma_db"
COLLECTION_NAME = "nz_police_manual"
EMBEDDING_MODEL = "nomic-embed-text:latest"

client = chromadb.PersistentClient(path=DB_PATH)
emb = OllamaEmbeddings(model=EMBEDDING_MODEL)

current_dim = len(emb.embed_query("dimension check"))

try:
    col = client.get_collection(COLLECTION_NAME)
    data = col.get(limit=1, include=["embeddings"])
    existing = data.get("embeddings") or []
    existing_dim = len(existing[0]) if existing and existing[0] else None
except Exception:
    col = None
    existing_dim = None

print({
    "collection": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "current_dim": current_dim,
    "existing_dim": existing_dim,
    "match": existing_dim is None or existing_dim == current_dim
})

{'collection': 'nz_police_manual', 'embedding_model': 'nomic-embed-text:latest', 'current_dim': 768, 'existing_dim': None, 'match': True}


In [38]:
pkill -f "uvicorn|fastapi|api" || true
sleep 2
cd /workspace/nz_legal_rag
uvicorn api.main:app --host 127.0.0.1 --port 8000 --reload

SyntaxError: invalid syntax (1430292361.py, line 1)

In [39]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/web/upload_page.py")
text = path.read_text(encoding="utf-8")

text = text.replace(
'''    collection = st.selectbox(
        "Target Collection",
        ["user_uploads", "Temporary Session Storage"],
        key="upload_collection_select"
    )
    
    is_temporary = collection == "Temporary Session Storage"
''',
'''    collection = st.selectbox(
        "Target Collection",
        ["nz_police_manual", "user_uploads", "Temporary Session Storage"],
        key="upload_collection_select"
    )
    
    is_temporary = collection == "Temporary Session Storage"
''')

text = text.replace(
'''                upload_multiple_files(uploaded_files, is_temporary)
''',
'''                upload_multiple_files(uploaded_files, collection, is_temporary)
''')

text = text.replace(
'''    collection = st.selectbox(
        "Target Collection",
        ["user_uploads", "Temporary Session Storage"],
        key="zip_collection_select"
    )
    
    is_temporary = collection == "Temporary Session Storage"
''',
'''    collection = st.selectbox(
        "Target Collection",
        ["nz_police_manual", "user_uploads", "Temporary Session Storage"],
        key="zip_collection_select"
    )
    
    is_temporary = collection == "Temporary Session Storage"
''')

text = text.replace(
'''                upload_zip_file(zip_file, is_temporary)
''',
'''                upload_zip_file(zip_file, collection, is_temporary)
''')

text = text.replace(
'''def upload_multiple_files(files: List[Any], is_temporary: bool):
''',
'''def upload_multiple_files(files: List[Any], collection: str, is_temporary: bool):
''')

text = text.replace(
'''        data = {"collection": "temp_session" if is_temporary else "user_uploads"}
''',
'''        data = {"collection": "temp_session" if is_temporary else collection}
''')

text = text.replace(
'''def upload_zip_file(zip_file: Any, is_temporary: bool):
''',
'''def upload_zip_file(zip_file: Any, collection: str, is_temporary: bool):
''')

text = text.replace(
'''        data = {"collection": "temp_session" if is_temporary else "user_uploads"}
''',
'''        data = {"collection": "temp_session" if is_temporary else collection}
''')

path.write_text(text, encoding="utf-8")
print(f"Patched {path}")

Patched /workspace/nz_legal_rag/web/upload_page.py


In [40]:
import subprocess, os
subprocess.run("pkill -f 'streamlit run web/streamlit_app' || true", shell=True, cwd="/workspace/nz_legal_rag")
subprocess.Popen(
    "/workspace/nz_legal_rag/.venv/bin/streamlit run web/streamlit_app",
    shell=True,
    cwd="/workspace/nz_legal_rag",
    stdout=open("/tmp/nzlr_streamlit.log", "a"),
    stderr=open("/tmp/nzlr_streamlit.log", "a"),
)
print("Streamlit restarted")

Streamlit restarted


In [ ]:
import sys, os
from pathlib import Path

ROOT = Path("/workspace/nz_legal_rag")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from api.server import rag_engine

collection_name = "policemanuals"

if rag_engine is None:
    raise RuntimeError("rag_engine is not initialized. Restart the API first, then rerun this cell.")

deleted = False
error = None

try:
    rag_engine.client.delete_collection(name=collection_name)
    deleted = True
except Exception as e:
    error = f"client.delete_collection failed: {e}"

try:
    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]
except Exception as e:
    error = error or f"in-memory cache removal failed: {e}"

print({
    "collection": collection_name,
    "deleted": deleted,
    "error": error,
    "remaining": collection_name in getattr(rag_engine, "collections", {}),
})

In [41]:
import sys
from pathlib import Path

ROOT = Path("/workspace/nz_legal_rag")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from api.server import rag_engine

collection_name = "nz_police_manual"

if rag_engine is None:
    raise RuntimeError("rag_engine is not initialized. Restart the API first, then rerun this cell.")

result = {
    "collection": collection_name,
    "deleted": False,
    "in_memory_before": collection_name in getattr(rag_engine, "collections", {}),
    "error": None,
}

try:
    rag_engine.client.delete_collection(name=collection_name)
    result["deleted"] = True
except Exception as e:
    result["error"] = f"client.delete_collection failed: {e}"

try:
    if collection_name in getattr(rag_engine, "collections", {}):
        del rag_engine.collections[collection_name]
except Exception as e:
    result["error"] = result["error"] or f"in-memory cache removal failed: {e}"

result["in_memory_after"] = collection_name in getattr(rag_engine, "collections", {})
print(result)

RuntimeError: rag_engine is not initialized. Restart the API first, then rerun this cell.

In [42]:
import chromadb

DB_PATH = "/workspace/chroma_db"
COLLECTION = "nz_police_manual"

client = chromadb.PersistentClient(path=DB_PATH)

before = [c.name for c in client.list_collections()]
deleted = False
error = None

try:
    client.delete_collection(name=COLLECTION)
    deleted = True
except Exception as e:
    error = str(e)

after = [c.name for c in client.list_collections()]

print({
    "db_path": DB_PATH,
    "collection": COLLECTION,
    "existed_before": COLLECTION in before,
    "deleted": deleted,
    "exists_after": COLLECTION in after,
    "error": error,
    "collections_before": before,
    "collections_after": after,
})

{'db_path': '/workspace/chroma_db', 'collection': 'nz_police_manual', 'existed_before': False, 'deleted': False, 'exists_after': False, 'error': 'Collection [nz_police_manual] does not exist', 'collections_before': ['user_uploads', 'temp_session_jVHlV-g00IgiMp20Qhl71FjZx0ngDuvWytW4ntk76Xw', 'temp_session_9a1b80gkv-EfSNTPToYDSWgc5MYQMdOfL8Ot6dgWLEA', 'temp_session_RlAQXjCpRfgQ4a4cY3hvCeLRE21Tj4FQQzqVvQfxqs8', 'temp_session_BKoYMVFYZl94HTbHzHZga8-xoDM6uemWymOxtpHQzq0', 'temp_session_knC7oOImluLGYQYtGRWW80B_BnvhXWIHJjYdoVtU-yc', 'temp_session_d-gk8b0Ki_kUrVrab-dUz-aUQNSwxm875xxiIwlPdHw', 'temp_session_5Lfa2f6z66e9k7wv9L-zQvwmxsTEAQn2_s02RixvEuc', 'session_fa98c3a7-96fa-4e67-ac77-57e9f847473d', 'temp_session_GTT8L5UIyHwZ8P6OAVp6Xs4WgU1vAH_CVuaUFuT18Qg', 'temp_session_zgGPat2wfvJ7fcRAVZueDCQ_Tsikh1dikHy0iIZN4pA', 'session_d723b93d-fd74-48ce-b5b6-0be63849a778', 'temp_session_EkAmKf0ead_D8G-o1-d2GDhG05T1AYovkXxpFRB0o3w', 'temp_session_LvGrmIxS_eaWttAhh_HvbBW8G5yIdkiRot-LwVoABW4', 'temp_sessio

In [43]:
import chromadb

DB_PATH = "/workspace/chroma_db"
COLLECTION = "police_manual"

client = chromadb.PersistentClient(path=DB_PATH)

before = [c.name for c in client.list_collections()]
deleted = False
error = None

try:
    client.delete_collection(name=COLLECTION)
    deleted = True
except Exception as e:
    error = str(e)

after = [c.name for c in client.list_collections()]

print({
    "db_path": DB_PATH,
    "collection": COLLECTION,
    "existed_before": COLLECTION in before,
    "deleted": deleted,
    "exists_after": COLLECTION in after,
    "error": error,
    "collections_before": before,
    "collections_after": after,
})

{'db_path': '/workspace/chroma_db', 'collection': 'police_manual', 'existed_before': True, 'deleted': True, 'exists_after': False, 'error': None, 'collections_before': ['user_uploads', 'temp_session_jVHlV-g00IgiMp20Qhl71FjZx0ngDuvWytW4ntk76Xw', 'temp_session_9a1b80gkv-EfSNTPToYDSWgc5MYQMdOfL8Ot6dgWLEA', 'temp_session_RlAQXjCpRfgQ4a4cY3hvCeLRE21Tj4FQQzqVvQfxqs8', 'temp_session_BKoYMVFYZl94HTbHzHZga8-xoDM6uemWymOxtpHQzq0', 'temp_session_knC7oOImluLGYQYtGRWW80B_BnvhXWIHJjYdoVtU-yc', 'temp_session_d-gk8b0Ki_kUrVrab-dUz-aUQNSwxm875xxiIwlPdHw', 'temp_session_5Lfa2f6z66e9k7wv9L-zQvwmxsTEAQn2_s02RixvEuc', 'session_fa98c3a7-96fa-4e67-ac77-57e9f847473d', 'temp_session_GTT8L5UIyHwZ8P6OAVp6Xs4WgU1vAH_CVuaUFuT18Qg', 'temp_session_zgGPat2wfvJ7fcRAVZueDCQ_Tsikh1dikHy0iIZN4pA', 'session_d723b93d-fd74-48ce-b5b6-0be63849a778', 'temp_session_EkAmKf0ead_D8G-o1-d2GDhG05T1AYovkXxpFRB0o3w', 'temp_session_LvGrmIxS_eaWttAhh_HvbBW8G5yIdkiRot-LwVoABW4', 'temp_session_4fIzwM9N0A56PCrQQOWpTqvK6kroqAV66xgpL7FXIV8',

In [44]:
import os, shutil
from pathlib import Path
import chromadb

DB_PATH = "/workspace/chroma_db"
SESSION_ROOT = Path("/workspace/temp_sessions")
TARGET_COLLECTION = "police_manual"

client = chromadb.PersistentClient(path=DB_PATH)

before = [c.name for c in client.list_collections()]
deleted_collection = False
deleted_dirs = []

try:
    client.delete_collection(name=TARGET_COLLECTION)
    deleted_collection = True
except Exception as e:
    print({"collection_delete_error": str(e)})

# Delete any session dirs that look related to that collection name
if SESSION_ROOT.exists():
    for child in SESSION_ROOT.iterdir():
        if child.is_dir() and TARGET_COLLECTION in child.name:
            try:
                shutil.rmtree(child, ignore_errors=True)
                deleted_dirs.append(child.name)
            except Exception as e:
                print({"dir_delete_error": child.name, "error": str(e)})

after = [c.name for c in client.list_collections()]

print({
    "db_path": DB_PATH,
    "collection": TARGET_COLLECTION,
    "deleted_collection": deleted_collection,
    "collections_before": before,
    "collections_after": after,
    "deleted_dirs": deleted_dirs,
})

{'collection_delete_error': 'Collection [police_manual] does not exist'}
{'db_path': '/workspace/chroma_db', 'collection': 'police_manual', 'deleted_collection': False, 'collections_before': ['user_uploads', 'temp_session_jVHlV-g00IgiMp20Qhl71FjZx0ngDuvWytW4ntk76Xw', 'temp_session_9a1b80gkv-EfSNTPToYDSWgc5MYQMdOfL8Ot6dgWLEA', 'temp_session_RlAQXjCpRfgQ4a4cY3hvCeLRE21Tj4FQQzqVvQfxqs8', 'temp_session_BKoYMVFYZl94HTbHzHZga8-xoDM6uemWymOxtpHQzq0', 'temp_session_knC7oOImluLGYQYtGRWW80B_BnvhXWIHJjYdoVtU-yc', 'temp_session_d-gk8b0Ki_kUrVrab-dUz-aUQNSwxm875xxiIwlPdHw', 'temp_session_5Lfa2f6z66e9k7wv9L-zQvwmxsTEAQn2_s02RixvEuc', 'session_fa98c3a7-96fa-4e67-ac77-57e9f847473d', 'temp_session_GTT8L5UIyHwZ8P6OAVp6Xs4WgU1vAH_CVuaUFuT18Qg', 'temp_session_zgGPat2wfvJ7fcRAVZueDCQ_Tsikh1dikHy0iIZN4pA', 'session_d723b93d-fd74-48ce-b5b6-0be63849a778', 'temp_session_EkAmKf0ead_D8G-o1-d2GDhG05T1AYovkXxpFRB0o3w', 'temp_session_LvGrmIxS_eaWttAhh_HvbBW8G5yIdkiRot-LwVoABW4', 'temp_session_4fIzwM9N0A56PCrQQOWpTq

In [46]:
import shutil
from pathlib import Path

SESSION_ROOT = Path("/workspace/temp_sessions")
deleted = []

if SESSION_ROOT.exists():
    for child in SESSION_ROOT.iterdir():
        if child.is_dir() and (child.name.startswith("temp_session_") or child.name.startswith("session_")):
            shutil.rmtree(child, ignore_errors=True)
            deleted.append(child.name)

print({"deleted_dirs": deleted, "count": len(deleted)})

{'deleted_dirs': [], 'count': 0}


In [47]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

models_anchor = """class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

models_insert = """class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]


class UploadFileResult(BaseModel):
    filename: str
    size_bytes: int
    status: str
    chunks_created: Optional[int] = None
    archive_path: Optional[str] = None
    notes: Optional[str] = None


class ConfirmationReport(BaseModel):
    report_type: str
    upload_id: str
    collection: str
    uploaded_at: str
    uploaded_by: str
    source_mode: str
    files_received: int
    files_ingested: int
    files_failed: int
    status: str
    file_results: List[UploadFileResult]
    embedding_model: Optional[str] = None
    vector_dimension: Optional[int] = None
    notes: List[str] = Field(default_factory=list)


class UploadConfirmationResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int
    confirmation_report: ConfirmationReport
"""

if "class UploadConfirmationResponse(BaseModel):" not in text:
    if models_anchor not in text:
        raise RuntimeError("Could not find FileUploadResponse block")
    text = text.replace(models_anchor, models_insert, 1)

text = text.replace(
    '@app.post("/api/v1/ingest/permanent", response_model=IngestResponse)',
    '@app.post("/api/v1/ingest/permanent", response_model=UploadConfirmationResponse)',
    1,
)

old_return = """    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
    }
"""

new_return = """    upload_id = f"perm_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{secrets.token_hex(4)}"

    file_results = []
    for doc in request.documents:
        content = doc.get("content", "") or ""
        filename = doc.get("filename", "untitled")
        file_results.append(
            {
                "filename": filename,
                "size_bytes": len(content.encode("utf-8")),
                "status": "ingested",
                "chunks_created": None,
                "archive_path": doc.get("archive_path"),
                "notes": None,
            }
        )

    files_received = len(request.documents)
    files_ingested = len(file_results)
    files_failed = max(0, files_received - files_ingested)

    status_value = (
        "accepted_to_permanent_collection"
        if files_ingested == files_received
        else "partial_success"
    )

    embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")
    vector_dimension = None
    try:
        if hasattr(rag_engine, "embedding_function"):
            probe = rag_engine.embedding_function.embed_query("dimension probe")
            vector_dimension = len(probe) if probe else None
    except Exception:
        vector_dimension = None

    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
        "confirmation_report": {
            "report_type": "permanent_collection_upload_confirmation",
            "upload_id": upload_id,
            "collection": target_collection,
            "uploaded_at": datetime.now().isoformat(),
            "uploaded_by": getattr(tenant, "username", "unknown"),
            "source_mode": "documents",
            "files_received": files_received,
            "files_ingested": files_ingested,
            "files_failed": files_failed,
            "status": status_value,
            "file_results": file_results,
            "embedding_model": embedding_model,
            "vector_dimension": vector_dimension,
            "notes": [],
        },
    }
"""

if old_return not in text:
    raise RuntimeError("Could not find old permanent ingest return block")

text = text.replace(old_return, new_return, 1)

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)
print("Inserted confirmation report models and patched /api/v1/ingest/permanent")

UPDATED: /workspace/nz_legal_rag/api/server.py
Inserted confirmation report models and patched /api/v1/ingest/permanent


In [48]:
import os
import subprocess
from pathlib import Path

project_root = Path("/workspace/nz_legal_rag")
os.chdir(project_root)

result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile api/server.py"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

RETURN CODE: 0
STDOUT:

STDERR:



In [49]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

old_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

new_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class UploadFileResult(BaseModel):
    filename: str
    size_bytes: int
    status: str
    chunks_created: Optional[int] = None
    archive_path: Optional[str] = None
    notes: Optional[str] = None


class ConfirmationReport(BaseModel):
    report_type: str
    upload_id: str
    collection: str
    uploaded_at: str
    uploaded_by: str
    source_mode: str
    files_received: int
    files_ingested: int
    files_failed: int
    status: str
    file_results: List[UploadFileResult]
    embedding_model: Optional[str] = None
    vector_dimension: Optional[int] = None
    notes: List[str] = Field(default_factory=list)


class UploadConfirmationResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int
    confirmation_report: ConfirmationReport


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

if "class UploadConfirmationResponse(BaseModel):" not in text:
    if old_models not in text:
        raise RuntimeError("Model anchor block not found in api/server.py")
    text = text.replace(old_models, new_models, 1)

old_decorator = '@app.post("/api/v1/ingest/permanent", response_model=IngestResponse)'
new_decorator = '@app.post("/api/v1/ingest/permanent", response_model=UploadConfirmationResponse)'

if old_decorator not in text:
    raise RuntimeError("Permanent ingest decorator not found")
text = text.replace(old_decorator, new_decorator, 1)

old_return = """    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
    }
"""

new_return = """    upload_id = f"perm_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{secrets.token_hex(4)}"

    file_results = []
    for doc in request.documents:
        content = doc.get("content", "") or ""
        filename = doc.get("filename") or doc.get("name") or "untitled"
        file_results.append(
            {
                "filename": filename,
                "size_bytes": len(content.encode("utf-8")),
                "status": "ingested",
                "chunks_created": None,
                "archive_path": doc.get("archive_path"),
                "notes": None,
            }
        )

    files_received = len(request.documents)
    files_ingested = len(file_results)
    files_failed = max(0, files_received - files_ingested)

    status_value = (
        "accepted_to_permanent_collection"
        if files_ingested == files_received
        else "partial_success"
    )

    embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")
    vector_dimension = None
    try:
        if hasattr(rag_engine, "embedding_function"):
            probe = rag_engine.embedding_function.embed_query("dimension probe")
            vector_dimension = len(probe) if probe else None
    except Exception:
        vector_dimension = None

    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
        "confirmation_report": {
            "report_type": "permanent_collection_upload_confirmation",
            "upload_id": upload_id,
            "collection": target_collection,
            "uploaded_at": datetime.now().isoformat(),
            "uploaded_by": getattr(tenant, "username", "unknown"),
            "source_mode": "documents",
            "files_received": files_received,
            "files_ingested": files_ingested,
            "files_failed": files_failed,
            "status": status_value,
            "file_results": file_results,
            "embedding_model": embedding_model,
            "vector_dimension": vector_dimension,
            "notes": [],
        },
    }
"""

permanent_start = text.find('@app.post("/api/v1/ingest/permanent", response_model=UploadConfirmationResponse)')
if permanent_start == -1:
    raise RuntimeError("Patched permanent ingest decorator not found")

temporary_start = text.find('@app.post("/api/v1/ingest/temporary"', permanent_start)
if temporary_start == -1:
    raise RuntimeError("Temporary ingest endpoint anchor not found")

permanent_block = text[permanent_start:temporary_start]
if old_return not in permanent_block:
    raise RuntimeError("Old permanent ingest return block not found inside permanent endpoint")

permanent_block = permanent_block.replace(old_return, new_return, 1)
text = text[:permanent_start] + permanent_block + text[temporary_start:]

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)
print("Patched models and /api/v1/ingest/permanent with confirmation_report")

RuntimeError: Permanent ingest decorator not found

In [50]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

old_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

new_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class UploadFileResult(BaseModel):
    filename: str
    size_bytes: int
    status: str
    chunks_created: Optional[int] = None
    archive_path: Optional[str] = None
    notes: Optional[str] = None


class ConfirmationReport(BaseModel):
    report_type: str
    upload_id: str
    collection: str
    uploaded_at: str
    uploaded_by: str
    source_mode: str
    files_received: int
    files_ingested: int
    files_failed: int
    status: str
    file_results: List[UploadFileResult]
    embedding_model: Optional[str] = None
    vector_dimension: Optional[int] = None
    notes: List[str] = Field(default_factory=list)


class UploadConfirmationResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int
    confirmation_report: ConfirmationReport


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

if "class UploadConfirmationResponse(BaseModel):" not in text:
    if old_models not in text:
        raise RuntimeError("Model anchor block not found")
    text = text.replace(old_models, new_models, 1)

perm_anchor = '@app.post("/api/v1/ingest/permanent"'
temp_anchor = '@app.post("/api/v1/ingest/temporary"'

perm_start = text.find(perm_anchor)
if perm_start == -1:
    raise RuntimeError("Permanent ingest endpoint anchor not found")

temp_start = text.find(temp_anchor, perm_start)
if temp_start == -1:
    raise RuntimeError("Temporary ingest endpoint anchor not found")

perm_block = text[perm_start:temp_start]

perm_block = perm_block.replace(
    'response_model=IngestResponse',
    'response_model=UploadConfirmationResponse',
    1,
)

old_return = """    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
    }
"""

new_return = """    upload_id = f"perm_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{secrets.token_hex(4)}"

    file_results = []
    for doc in request.documents:
        content = doc.get("content", "") or ""
        filename = doc.get("filename") or doc.get("name") or "untitled"
        file_results.append(
            {
                "filename": filename,
                "size_bytes": len(content.encode("utf-8")),
                "status": "ingested",
                "chunks_created": None,
                "archive_path": doc.get("archive_path"),
                "notes": None,
            }
        )

    files_received = len(request.documents)
    files_ingested = len(file_results)
    files_failed = max(0, files_received - files_ingested)

    status_value = (
        "accepted_to_permanent_collection"
        if files_ingested == files_received
        else "partial_success"
    )

    embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")
    vector_dimension = None
    try:
        if hasattr(rag_engine, "embedding_function"):
            probe = rag_engine.embedding_function.embed_query("dimension probe")
            vector_dimension = len(probe) if probe else None
    except Exception:
        vector_dimension = None

    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
        "confirmation_report": {
            "report_type": "permanent_collection_upload_confirmation",
            "upload_id": upload_id,
            "collection": target_collection,
            "uploaded_at": datetime.now().isoformat(),
            "uploaded_by": getattr(tenant, "username", "unknown"),
            "source_mode": "documents",
            "files_received": files_received,
            "files_ingested": files_ingested,
            "files_failed": files_failed,
            "status": status_value,
            "file_results": file_results,
            "embedding_model": embedding_model,
            "vector_dimension": vector_dimension,
            "notes": [],
        },
    }
"""

if old_return not in perm_block:
    raise RuntimeError("Old permanent return block not found inside permanent endpoint")

perm_block = perm_block.replace(old_return, new_return, 1)
text = text[:perm_start] + perm_block + text[temp_start:]

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)
print("Patched permanent ingest endpoint using route anchors")

RuntimeError: Old permanent return block not found inside permanent endpoint

In [51]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

old_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

new_models = """class IngestResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int


class UploadFileResult(BaseModel):
    filename: str
    size_bytes: int
    status: str
    chunks_created: Optional[int] = None
    archive_path: Optional[str] = None
    notes: Optional[str] = None


class ConfirmationReport(BaseModel):
    report_type: str
    upload_id: str
    collection: str
    uploaded_at: str
    uploaded_by: str
    source_mode: str
    files_received: int
    files_ingested: int
    files_failed: int
    status: str
    file_results: List[UploadFileResult]
    embedding_model: Optional[str] = None
    vector_dimension: Optional[int] = None
    notes: List[str] = Field(default_factory=list)


class UploadConfirmationResponse(BaseModel):
    success: bool
    message: str
    chunks_ingested: int
    confirmation_report: ConfirmationReport


class FileUploadResponse(BaseModel):
    success: bool
    message: str
    files_processed: int
    files_failed: int
    total_chunks: int
    details: List[Dict[str, Any]]
"""

if "class UploadConfirmationResponse(BaseModel):" not in text:
    if old_models not in text:
        raise RuntimeError("Model anchor block not found")
    text = text.replace(old_models, new_models, 1)

perm_anchor = '@app.post("/api/v1/ingest/permanent"'
temp_anchor = '@app.post("/api/v1/ingest/temporary"'

perm_start = text.find(perm_anchor)
if perm_start == -1:
    raise RuntimeError("Permanent ingest endpoint anchor not found")

temp_start = text.find(temp_anchor, perm_start)
if temp_start == -1:
    raise RuntimeError("Temporary ingest endpoint anchor not found")

perm_block = text[perm_start:temp_start]

perm_block = perm_block.replace(
    "response_model=IngestResponse",
    "response_model=UploadConfirmationResponse",
    1,
)

usage_anchor = """    if tenant_manager:
        tenant_manager.record_usage(
            tenant.tenant_id,
            storage_bytes=sum(len(d.get("content", "")) for d in request.documents),
            api_calls=1,
        )
"""

usage_pos = perm_block.find(usage_anchor)
if usage_pos == -1:
    raise RuntimeError("Usage-record block not found in permanent endpoint")

replacement_tail = """    if tenant_manager:
        tenant_manager.record_usage(
            tenant.tenant_id,
            storage_bytes=sum(len(d.get("content", "")) for d in request.documents),
            api_calls=1,
        )

    upload_id = f"perm_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{secrets.token_hex(4)}"

    file_results = []
    for doc in request.documents:
        content = doc.get("content", "") or ""
        filename = doc.get("filename") or doc.get("name") or "untitled"
        file_results.append(
            {
                "filename": filename,
                "size_bytes": len(content.encode("utf-8")),
                "status": "ingested",
                "chunks_created": None,
                "archive_path": doc.get("archive_path"),
                "notes": None,
            }
        )

    files_received = len(request.documents)
    files_ingested = len(file_results)
    files_failed = max(0, files_received - files_ingested)

    status_value = (
        "accepted_to_permanent_collection"
        if files_ingested == files_received
        else "partial_success"
    )

    embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")
    vector_dimension = None
    try:
        if hasattr(rag_engine, "embedding_function"):
            probe = rag_engine.embedding_function.embed_query("dimension probe")
            vector_dimension = len(probe) if probe else None
    except Exception:
        vector_dimension = None

    return {
        "success": True,
        "message": msg,
        "chunks_ingested": chunks,
        "confirmation_report": {
            "report_type": "permanent_collection_upload_confirmation",
            "upload_id": upload_id,
            "collection": target_collection,
            "uploaded_at": datetime.now().isoformat(),
            "uploaded_by": getattr(tenant, "username", "unknown"),
            "source_mode": "documents",
            "files_received": files_received,
            "files_ingested": files_ingested,
            "files_failed": files_failed,
            "status": status_value,
            "file_results": file_results,
            "embedding_model": embedding_model,
            "vector_dimension": vector_dimension,
            "notes": [],
        },
    }


"""

perm_block = perm_block[:usage_pos] + replacement_tail
text = text[:perm_start] + perm_block + text[temp_start:]

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)
print("Patched permanent ingest endpoint using usage-record anchor")

UPDATED: /workspace/nz_legal_rag/api/server.py
Patched permanent ingest endpoint using usage-record anchor


In [52]:
import os
import subprocess
from pathlib import Path

project_root = Path("/workspace/nz_legal_rag")
os.chdir(project_root)

result = subprocess.run(
    ["bash", "-lc", "python3 -m py_compile api/server.py"],
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

RETURN CODE: 0
STDOUT:

STDERR:



In [53]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

helper_anchor = """def login() -> None:
"""
helper_block = """def render_stickman_loader(message: str = "Working behind the scenes...") -> None:
    st.markdown(
        f\"\"\"
        <div class="aegis-loader-wrap">
          <div class="aegis-loader-card">
            <div class="aegis-loader-stage">{message}</div>
            <div class="aegis-loader-scene" aria-label="{message}">
              <svg class="aegis-loader-svg" viewBox="0 0 240 180" role="img" aria-hidden="true">
                <g class="aegis-loader-shadow">
                  <ellipse cx="120" cy="154" rx="42" ry="10"></ellipse>
                </g>
                <g class="aegis-loader-stickman">
                  <circle class="head" cx="120" cy="42" r="16"></circle>
                  <line class="body" x1="120" y1="58" x2="120" y2="102"></line>
                  <line class="arm arm-left" x1="120" y1="72" x2="92" y2="92"></line>
                  <line class="arm arm-right" x1="120" y1="72" x2="148" y2="92"></line>
                  <line class="leg leg-left" x1="120" y1="102" x2="98" y2="142"></line>
                  <line class="leg leg-right" x1="120" y1="102" x2="142" y2="142"></line>
                  <g class="briefcase">
                    <rect x="150" y="90" width="26" height="20" rx="3"></rect>
                    <line x1="158" y1="90" x2="158" y2="84"></line>
                    <line x1="168" y1="90" x2="168" y2="84"></line>
                    <line x1="158" y1="84" x2="168" y2="84"></line>
                  </g>
                </g>
                <g class="aegis-loader-dots">
                  <circle cx="60" cy="52" r="4"></circle>
                  <circle cx="76" cy="52" r="4"></circle>
                  <circle cx="92" cy="52" r="4"></circle>
                </g>
              </svg>
            </div>
            <div class="aegis-loader-caption">Please wait while AEGIS handles the background work.</div>
          </div>
        </div>
        \"\"\",
        unsafe_allow_html=True,
    )


def run_with_stickman(message: str, fn):
    loader_slot = st.empty()
    try:
        with loader_slot.container():
            render_stickman_loader(message)
        with st.spinner(message, show_time=True):
            return fn()
    finally:
        loader_slot.empty()


def login() -> None:
"""

if "def run_with_stickman(message: str, fn):" not in text:
    if helper_anchor not in text:
        raise RuntimeError("Could not find login() anchor for loader helper insertion")
    text = text.replace(helper_anchor, helper_block, 1)

old_cm = """            with st.spinner("Uploading files into the permanent collection..."):
                if len(uploaded_files) == 1 and uploaded_files[0].name.lower().endswith(".zip"):
                    result = api_call(
                        "/api/v1/upload/zip",
                        data={"collection": collection_name},
                        method="POST",
                        api_key=st.session_state.get("api_key"),
                        session_id=st.session_state.get("session_id"),
                        files=uploaded_files,
                        file_field_name="file",
                    )
                else:
                    result = api_call(
                        "/api/v1/upload/files",
                        data={"collection": collection_name},
                        method="POST",
                        api_key=st.session_state.get("api_key"),
                        session_id=st.session_state.get("session_id"),
                        files=uploaded_files,
                    )
"""

new_cm = """            def _cm_upload():
                if len(uploaded_files) == 1 and uploaded_files[0].name.lower().endswith(".zip"):
                    return api_call(
                        "/api/v1/upload/zip",
                        data={"collection": collection_name},
                        method="POST",
                        api_key=st.session_state.get("api_key"),
                        session_id=st.session_state.get("session_id"),
                        files=uploaded_files,
                        file_field_name="file",
                    )
                return api_call(
                    "/api/v1/upload/files",
                    data={"collection": collection_name},
                    method="POST",
                    api_key=st.session_state.get("api_key"),
                    session_id=st.session_state.get("session_id"),
                    files=uploaded_files,
                )

            result = run_with_stickman(
                "Uploading files into the permanent collection...",
                _cm_upload,
            )
"""

if old_cm not in text:
    raise RuntimeError("Collection Manager upload block not found")
text = text.replace(old_cm, new_cm, 1)

old_search_1 = """        with st.spinner("Researching and explaining..."):
            analysis_result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query_text,
                    "analysis_type": "general",
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )
"""

new_search_1 = """        def _search_analysis():
            return api_call(
                "/api/v1/analyze",
                data={
                    "query": query_text,
                    "analysis_type": "general",
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        analysis_result = run_with_stickman(
            "Researching and explaining...",
            _search_analysis,
        )
"""

if old_search_1 not in text:
    raise RuntimeError("Search analysis spinner block not found")
text = text.replace(old_search_1, new_search_1, 1)

old_search_2 = """        with st.spinner("Loading verified source list..."):
            search_result = api_call(
                "/api/v1/search",
                data={"query": query_text},
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )
"""

new_search_2 = """        def _search_sources():
            return api_call(
                "/api/v1/search",
                data={"query": query_text},
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
            )

        search_result = run_with_stickman(
            "Loading verified source list...",
            _search_sources,
        )
"""

if old_search_2 not in text:
    raise RuntimeError("Search sources spinner block not found")
text = text.replace(old_search_2, new_search_2, 1)

old_analysis = """        with st.spinner("Analysing..."):
            result = api_call(
                "/api/v1/analyze",
                data={
                    "query": query.strip(),
                    "analysis_type": analysis_type,
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
                connect_timeout=15,
                read_timeout=180,
            )
"""

new_analysis = """        def _run_analysis():
            return api_call(
                "/api/v1/analyze",
                data={
                    "query": query.strip(),
                    "analysis_type": analysis_type,
                },
                method="POST",
                api_key=st.session_state.get("api_key"),
                session_id=st.session_state.get("session_id"),
                connect_timeout=15,
                read_timeout=180,
            )

        result = run_with_stickman(
            "Analysing legal materials...",
            _run_analysis,
        )
"""

if old_analysis not in text:
    raise RuntimeError("Analysis spinner block not found")
text = text.replace(old_analysis, new_analysis, 1)

old_admin = """    with col_a:
        if st.button("Load Database Stats", use_container_width=True, key="admin_load_db_stats"):
            stats = api_call(
                "/health/deep",
                method="GET",
                session_id=st.session_state["session_id"],
            )
            if stats:
                st.session_state["admin_db_stats"] = stats
"""

new_admin = """    with col_a:
        if st.button("Load Database Stats", use_container_width=True, key="admin_load_db_stats"):
            def _load_db_stats():
                return api_call(
                    "/health/deep",
                    method="GET",
                    session_id=st.session_state["session_id"],
                )

            stats = run_with_stickman(
                "Loading database statistics...",
                _load_db_stats,
            )
            if stats:
                st.session_state["admin_db_stats"] = stats
"""

if old_admin not in text:
    raise RuntimeError("Admin stats load block not found")
text = text.replace(old_admin, new_admin, 1)

path.write_text(text, encoding="utf-8")
print("UPDATED:", path)
print("Inserted reusable stickman loader and wired it into Collection Manager, Search, Analysis, and Admin Panel")

UPDATED: /workspace/nz_legal_rag/web/streamlit_app.py
Inserted reusable stickman loader and wired it into Collection Manager, Search, Analysis, and Admin Panel


In [54]:
import os
import subprocess

os.chdir("/workspace/nz_legal_rag")
subprocess.run(["python3", "-m", "py_compile", "web/streamlit_app.py"], check=False)

CompletedProcess(args=['python3', '-m', 'py_compile', 'web/streamlit_app.py'], returncode=0)

In [55]:
# Notebook cell: bump admin stats timeout from 60s to 180s
from pathlib import Path

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

old_block = """    with col_a:
        if st.button("Load Database Stats", use_container_width=True, key="admin_load_db_stats"):
            stats = api_call(
                "/health/deep",
                method="GET",
                session_id=st.session_state["session_id"],
            )
            if stats:
                st.session_state["admin_db_stats"] = stats
"""

new_block = """    with col_a:
        if st.button("Load Database Stats", use_container_width=True, key="admin_load_db_stats"):
            def _load_db_stats():
                return api_call(
                    "/health/deep",
                    method="GET",
                    session_id=st.session_state["session_id"],
                    connect_timeout=15,
                    read_timeout=180,
                )

            stats = run_with_stickman(
                "Loading database statistics...",
                _load_db_stats,
            )
            if stats:
                st.session_state["admin_db_stats"] = stats
"""

if old_block not in text:
    raise RuntimeError("Could not find the admin Load Database Stats block")

text = text.replace(old_block, new_block, 1)
path.write_text(text, encoding="utf-8")

print("PATCHED: web/streamlit_app.py")
print("Added read_timeout=180 to /health/deep call in show_admin_panel()")

RuntimeError: Could not find the admin Load Database Stats block

In [56]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

pattern = re.compile(
    r'(return api_call\(\s*'
    r'"/health/deep",\s*'
    r'method="GET",\s*'
    r'session_id=st\.session_state\["session_id"\],\s*)'
    r'((?:connect_timeout=\d+,\s*)?)'
    r'((?:read_timeout=\d+,\s*)?)'
    r'(\s*\))',
    re.MULTILINE
)

match = pattern.search(text)
if not match:
    raise RuntimeError("Could not find /health/deep api_call block")

replacement = (
    match.group(1)
    + 'connect_timeout=15,\n'
    + '                    read_timeout=180,\n'
    + match.group(4)
)

new_text = text[:match.start()] + replacement + text[match.end():]
path.write_text(new_text, encoding="utf-8")

print("PATCHED:", path)
print("Set /health/deep admin stats call to connect_timeout=15, read_timeout=180")

PATCHED: /workspace/nz_legal_rag/web/streamlit_app.py
Set /health/deep admin stats call to connect_timeout=15, read_timeout=180


In [57]:
from pathlib import Path
import subprocess

path = Path("/workspace/nz_legal_rag/web/streamlit_app.py")
text = path.read_text(encoding="utf-8")

start = text.find('"/health/deep"')
print(text[start-120:start+220])

subprocess.run(
    ["python3", "-m", "py_compile", str(path)],
    check=False,
)

rue, key="admin_load_db_stats"):
            def _load_db_stats():
                return api_call(
                    "/health/deep",
                    method="GET",
                    session_id=st.session_state["session_id"],
                connect_timeout=15,
                    read_timeout=180,
)

            stats = run_with_s


CompletedProcess(args=['python3', '-m', 'py_compile', '/workspace/nz_legal_rag/web/streamlit_app.py'], returncode=0)

In [59]:
import requests, json

base = "http://127.0.0.1:8000"

for path in ["/health/deep", "/api/v1/collections"]:
    r = requests.get(base + path, timeout=(15, 180))
    print("\n==", path, r.status_code, "==")
    try:
        data = r.json()
        print(json.dumps(data, indent=2)[:8000])
    except Exception:
        print(r.text[:4000])


== /health/deep 200 ==
{
  "status": "healthy",
  "service": "nz-legal-rag-api",
  "database": {
    "status": "initialized",
    "path": "/workspace/chroma_db",
    "documents": 433234,
    "chunks": 444120,
    "stats": {
      "collections": {
        "nz_legislation": {
          "count": 432854,
          "documents": 432854,
          "description": "NZ Legislation (Acts, Regulations)"
        },
        "nzlii_criminal_cases": {
          "count": 9981,
          "documents": 370,
          "description": "NZ Case Law"
        },
        "user_uploads": {
          "count": 1285,
          "documents": 10,
          "description": "User Uploaded Documents"
        }
      },
      "total_documents": 433234,
      "total_chunks": 444120
    },
    "readiness": "ready"
  },
  "tenants": 3,
  "timestamp": "2026-06-14T16:15:21.562574"
}

== /api/v1/collections 200 ==
{
  "collections": [
    {
      "id": "nz_legal_unified",
      "description": "NZ Legal Unified Database",
      "

In [60]:
import chromadb
from pprint import pprint

CHROMA_DB_PATH = "/workspace/chroma_db"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

def safe_count(name: str):
    try:
        return client.get_collection(name).count()
    except Exception as e:
        return f"ERROR: {e}"

def exists(name: str) -> bool:
    try:
        client.get_collection(name)
        return True
    except Exception:
        return False

print("Before:")
for c in client.list_collections():
    name = getattr(c, "name", str(c))
    print(f" - {name}: {safe_count(name)}")

# 1) Rename nzlii_criminal_cases -> nz_case_law
old_name = "nzlii_criminal_cases"
new_name = "nz_case_law"

if exists(old_name):
    if exists(new_name):
        raise RuntimeError(
            f"Cannot rename {old_name} to {new_name}: target already exists. "
            "Delete or rename the target first."
        )
    col = client.get_collection(old_name)
    col.modify(name=new_name)
    print(f"Renamed {old_name} -> {new_name}")
else:
    print(f"Skip rename: {old_name} not found")

# 2) Delete user_uploads
for name in ["user_uploads", "nz_legal_unified"]:
    if exists(name):
        client.delete_collection(name=name)
        print(f"Deleted {name}")
    else:
        print(f"Skip delete: {name} not found")

print("\nAfter:")
remaining = []
for c in client.list_collections():
    name = getattr(c, "name", str(c))
    remaining.append({"name": name, "count": safe_count(name)})
    print(f" - {name}: {safe_count(name)}")

print("\nRemaining collections:")
pprint(remaining)

Before:
 - user_uploads: 1285
 - temp_session_jVHlV-g00IgiMp20Qhl71FjZx0ngDuvWytW4ntk76Xw: 178
 - temp_session_9a1b80gkv-EfSNTPToYDSWgc5MYQMdOfL8Ot6dgWLEA: 9
 - temp_session_RlAQXjCpRfgQ4a4cY3hvCeLRE21Tj4FQQzqVvQfxqs8: 28
 - temp_session_BKoYMVFYZl94HTbHzHZga8-xoDM6uemWymOxtpHQzq0: 9
 - temp_session_knC7oOImluLGYQYtGRWW80B_BnvhXWIHJjYdoVtU-yc: 178
 - temp_session_d-gk8b0Ki_kUrVrab-dUz-aUQNSwxm875xxiIwlPdHw: 178
 - temp_session_5Lfa2f6z66e9k7wv9L-zQvwmxsTEAQn2_s02RixvEuc: 0
 - session_fa98c3a7-96fa-4e67-ac77-57e9f847473d: 9
 - temp_session_GTT8L5UIyHwZ8P6OAVp6Xs4WgU1vAH_CVuaUFuT18Qg: 178
 - temp_session_zgGPat2wfvJ7fcRAVZueDCQ_Tsikh1dikHy0iIZN4pA: 9
 - session_d723b93d-fd74-48ce-b5b6-0be63849a778: 261
 - temp_session_EkAmKf0ead_D8G-o1-d2GDhG05T1AYovkXxpFRB0o3w: 9
 - temp_session_LvGrmIxS_eaWttAhh_HvbBW8G5yIdkiRot-LwVoABW4: 9
 - temp_session_4fIzwM9N0A56PCrQQOWpTqvK6kroqAV66xgpL7FXIV8: 178
 - temp_session_8_2BeroylLwnQqZ2eKkRxKDYRtYaXbNPAlesOhl74XY: 9
 - nz_legislation: 432854
 - temp_se

In [61]:
import requests, json
r = requests.get("http://127.0.0.1:8000/api/v1/collections", timeout=(15, 180))
print(json.dumps(r.json(), indent=2))

{
  "collections": [
    {
      "id": "nz_legal_unified",
      "description": "NZ Legal Unified Database",
      "document_count": 0
    },
    {
      "id": "nz_legislation",
      "description": "NZ Legislation (Acts, Regulations)",
      "document_count": 432854
    },
    {
      "id": "nz_case_law",
      "description": "NZ Case Law",
      "document_count": 0
    },
    {
      "id": "nzlii_criminal_cases",
      "description": "NZ Case Law",
      "document_count": 9981
    },
    {
      "id": "nz_police_manual",
      "description": "NZ Police Manual",
      "document_count": 0
    },
    {
      "id": "legal_research",
      "description": "Legal Research",
      "document_count": 0
    },
    {
      "id": "user_uploads",
      "description": "User Uploaded Documents",
      "document_count": 0
    },
    {
      "id": "confidential",
      "description": "Confidential Documents (Local)",
      "document_count": 0
    }
  ]
}


In [62]:
import chromadb

CHROMA_DB_PATH = "/workspace/chroma_db"
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

all_names = []
for c in client.list_collections():
    name = getattr(c, "name", None) or str(c)
    all_names.append(name)

targets = sorted([name for name in all_names if "session" in name.lower()])

print("Collections to delete:")
for name in targets:
    print(" -", name)

if not targets:
    print("No session collections found.")
else:
    deleted = []
    failed = []

    for name in targets:
        try:
            client.delete_collection(name=name)
            deleted.append(name)
        except Exception as e:
            failed.append((name, str(e)))

    print("\nDeleted:")
    for name in deleted:
        print(" -", name)

    if failed:
        print("\nFailed:")
        for name, err in failed:
            print(f" - {name}: {err}")

print("\nRemaining collections:")
for c in client.list_collections():
    name = getattr(c, "name", None) or str(c)
    try:
        count = client.get_collection(name).count()
    except Exception as e:
        count = f"ERROR: {e}"
    print(f" - {name}: {count}")

Collections to delete:
 - session_69a0de06-162d-4f0e-802d-cf0e65d5bc20
 - session_d723b93d-fd74-48ce-b5b6-0be63849a778
 - session_dc905481-e9cc-4033-b149-d0e4397dc5e9
 - session_fa98c3a7-96fa-4e67-ac77-57e9f847473d
 - temp_session_-inW5G5mfzM4rBxv0XVHE8ynopSFbtb0pXMANWSK_rE
 - temp_session_05b739c0-2d6a-4ba7-9124-6c2c9160c04b
 - temp_session_0c0402fb-23a1-4200-9e8e-ee78037f4d73
 - temp_session_2Weam9vqKrW8pQx5VV9KEw3vLcBlXSvhRk8vfxkl2g4
 - temp_session_4fIzwM9N0A56PCrQQOWpTqvK6kroqAV66xgpL7FXIV8
 - temp_session_5Lfa2f6z66e9k7wv9L-zQvwmxsTEAQn2_s02RixvEuc
 - temp_session_5qoIKyVkeZ0hT7m_OCX2F914jznNAj7SxDuBBVyRBos
 - temp_session_63feb867-9162-4905-b26a-629806c74a7b
 - temp_session_8HYVx_EoaZx4M-fkirFREJmzs-NkYgg4FtaOMI9uFcQ
 - temp_session_8_2BeroylLwnQqZ2eKkRxKDYRtYaXbNPAlesOhl74XY
 - temp_session_9a1b80gkv-EfSNTPToYDSWgc5MYQMdOfL8Ot6dgWLEA
 - temp_session_BKoYMVFYZl94HTbHzHZga8-xoDM6uemWymOxtpHQzq0
 - temp_session_DeZ9zmV95UF-Am1R5pzQ3f1pS-iBmyUPcJZW836fSiI
 - temp_session_DwZDpgf3Kj

In [63]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

# Target the collection_map list in the patched route
old_map = '''
        collection_map = [
            {
                "id": "nz_legislation",
                "description": "NZ Legislation (Acts, Regulations)",
            },
            {
                "id": "nzlii_criminal_cases",
                "description": "NZ Case Law",
            },
            {
                "id": "NZ_Police_Manuals",
                "description": "NZ Police Manuals",
            },
        ]
'''

new_map = '''
        collection_map = [
            {
                "id": "nz_legislation",
                "description": "NZ Legislation",
            },
            {
                "id": "nzlii_criminal_cases",
                "description": "NZ Case Law",
            },
            {
                "id": "NZ_Police_Manuals",
                "description": "NZ Police Manuals",
            },
        ]
'''

if old_map not in text:
    raise RuntimeError("Target collection_map not found in api/server.py")

text = text.replace(old_map, new_map, 1)
path.write_text(text, encoding="utf-8")

print("PATCHED:", path)
print("Changed nz_legislation description to 'NZ Legislation'")

RuntimeError: Target collection_map not found in api/server.py

In [64]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

pattern = re.compile(
    r'("id"\s*:\s*"nz_legislation"\s*,\s*"description"\s*:\s*")([^"]+)(")',
    re.S
)

new_text, n = pattern.subn(r'\1NZ Legislation\3', text, count=1)

if n == 0:
    raise RuntimeError('Could not find nz_legislation description block in api/server.py')

path.write_text(new_text, encoding="utf-8")
print("PATCHED:", path)
print('Changed nz_legislation description to "NZ Legislation"')

RuntimeError: Could not find nz_legislation description block in api/server.py

In [65]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

for needle in [
    'nz_legislation',
    '/api/v1/collections',
    'NZ Legislation',
    'Acts, Regulations',
]:
    print(f"\n--- searching for: {needle} ---")
    idx = text.find(needle)
    print("index:", idx)
    if idx != -1:
        start = max(0, idx - 300)
        end = min(len(text), idx + 500)
        print(text[start:end])


--- searching for: nz_legislation ---
index: 14320
n(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"This endpoint requires one of: {[r.value for r in roles]}",
            )
        return tenant
    return checker




LEGACY_TEMP_PREFIX = "temp_"
TEMP_SESSION_PREFIX = "temp_session_"
PERMANENT_COLLECTIONS = {
    "nz_legislation",
    "nz_case_law",
    "nzlii_criminal_cases",
    "nz_police_manual",
}

def is_deletable_collection(name: str) -> bool:
    if not name:
        return False
    if name in PERMANENT_COLLECTIONS:
        return False
    return (
        name == "user_uploads"
        or name.startswith(LEGACY_TEMP_PREFIX)
        or name.startswith(TEMP_SESSION_PREFIX)
    )
@app.get("/")
def root():
    return {
        "name": "NZ Legal RAG API",
        "version": "1.2.2",
        "status"

--- searching for: /api/v1/collections ---
index: 20333
 "ok": True,
        "access_token": api_key,
        "session_id": session_id,
        "d

In [66]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/api/server.py")
text = path.read_text(encoding="utf-8")

new_text = text

# Case 1: simple literal replacement if long label still exists
new_text = new_text.replace("NZ Legislation (Acts, Regulations)", "NZ Legislation")

# Case 2: regex fallback for dict-style entry near nz_legislation
patterns = [
    re.compile(
        r'(\{\s*"id"\s*:\s*"nz_legislation"\s*,\s*"description"\s*:\s*")([^"]+)(")',
        re.S,
    ),
    re.compile(
        r"(\{\s*'id'\s*:\s*'nz_legislation'\s*,\s*'description'\s*:\s*')([^']+)(')",
        re.S,
    ),
]

changed = (new_text != text)

if not changed:
    for pattern in patterns:
        candidate, n = pattern.subn(r'\1NZ Legislation\3', new_text, count=1)
        if n:
            new_text = candidate
            changed = True
            break

if not changed:
    raise RuntimeError("Could not find a replaceable nz_legislation label in api/server.py")

path.write_text(new_text, encoding="utf-8")
print("PATCHED:", path)
print('Normalized label to "NZ Legislation"')

RuntimeError: Could not find a replaceable nz_legislation label in api/server.py

In [67]:
from pathlib import Path

candidates = [
    Path("/workspace/nz_legal_rag/api/server.py"),
    Path("api/server.py"),
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise RuntimeError("Could not find api/server.py")

print("Using:", path)

text = path.read_text(encoding="utf-8")
lines = text.splitlines()

needles = ["/api/v1/collections", "nz_legislation", "NZ Legislation", "Acts, Regulations"]
hits = []

for i, line in enumerate(lines, start=1):
    if any(n in line for n in needles):
        hits.append(i)

if not hits:
    raise RuntimeError("No matching lines found in api/server.py")

for ln in hits[:20]:
    start = max(1, ln - 4)
    end = min(len(lines), ln + 8)
    print("\n" + "=" * 80)
    print(f"Lines {start}-{end}")
    print("=" * 80)
    for j in range(start, end + 1):
        print(f"{j:4}: {lines[j-1]}")

Using: /workspace/nz_legal_rag/api/server.py

Lines 514-526
 514: 
 515: LEGACY_TEMP_PREFIX = "temp_"
 516: TEMP_SESSION_PREFIX = "temp_session_"
 517: PERMANENT_COLLECTIONS = {
 518:     "nz_legislation",
 519:     "nz_case_law",
 520:     "nzlii_criminal_cases",
 521:     "nz_police_manual",
 522: }
 523: 
 524: def is_deletable_collection(name: str) -> bool:
 525:     if not name:
 526:         return False

Lines 702-714
 702:         "detail": detail,
 703:     }
 704: 
 705: 
 706: @app.get("/api/v1/collections")
 707: def list_collections():
 708:     if not rag_engine:
 709:         raise HTTPException(status_code=500, detail="RAG engine not initialized")
 710: 
 711:     collections = []
 712:     for name, desc in NZLegalRAG.COLLECTIONS.items():
 713:         try:
 714:             count = rag_engine.collections.get(name, {}).count() if name in rag_engine.collections else 0


In [68]:
from pathlib import Path

candidates = [
    Path("/workspace/nz_legal_rag/core/rag_engine.py"),
    Path("core/rag_engine.py"),
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise RuntimeError("Could not find core/rag_engine.py")

print("Using:", path)

text = path.read_text(encoding="utf-8")
lines = text.splitlines()

hits = []
for i, line in enumerate(lines, start=1):
    if "COLLECTIONS" in line or "nz_legislation" in line or "Acts, Regulations" in line or "NZ Legislation" in line:
        hits.append(i)

for ln in hits[:20]:
    start = max(1, ln - 4)
    end = min(len(lines), ln + 8)
    print("\n" + "=" * 80)
    print(f"Lines {start}-{end}")
    print("=" * 80)
    for j in range(start, end + 1):
        print(f"{j:4}: {lines[j-1]}")

Using: /workspace/nz_legal_rag/core/rag_engine.py

Lines 63-75
  63:     - Citation tracking
  64:     - Confidence scoring
  65:     """
  66:     
  67:     COLLECTIONS = {
  68:         "nz_legal_unified": "NZ Legal Unified Database",
  69:         "nz_legislation": "NZ Legislation (Acts, Regulations)",
  70:         "nz_case_law": "NZ Case Law",
  71:         "nzlii_criminal_cases": "NZ Case Law",
  72:         "nz_police_manual": "NZ Police Manual",
  73:         "legal_research": "Legal Research",
  74:         "user_uploads": "User Uploaded Documents",
  75:         "confidential": "Confidential Documents (Local)"

Lines 65-77
  65:     """
  66:     
  67:     COLLECTIONS = {
  68:         "nz_legal_unified": "NZ Legal Unified Database",
  69:         "nz_legislation": "NZ Legislation (Acts, Regulations)",
  70:         "nz_case_law": "NZ Case Law",
  71:         "nzlii_criminal_cases": "NZ Case Law",
  72:         "nz_police_manual": "NZ Police Manual",
  73:         "legal_re

In [69]:
from pathlib import Path

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text(encoding="utf-8")

old = '"nz_legislation": "NZ Legislation (Acts, Regulations)"'
new = '"nz_legislation": "NZ Legislation"'

if old not in text:
    raise RuntimeError("Exact nz_legislation label not found in core/rag_engine.py")

text = text.replace(old, new, 1)
path.write_text(text, encoding="utf-8")

print("PATCHED:", path)
print('Changed nz_legislation label to "NZ Legislation"')

PATCHED: /workspace/nz_legal_rag/core/rag_engine.py
Changed nz_legislation label to "NZ Legislation"


In [70]:
import os
import subprocess
import time
import requests
import json

os.chdir("/workspace/nz_legal_rag")

subprocess.run(["python3", "-m", "py_compile", "core/rag_engine.py", "api/server.py"], check=False)
subprocess.run(["pkill", "-f", "uvicorn api.server:app"], check=False)

subprocess.Popen(
    ["/workspace/nz_legal_rag/.venv/bin/python", "-m", "uvicorn", "api.server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/tmp/uvicorn.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(4)

for endpoint in ["/api/v1/collections", "/health/deep"]:
    r = requests.get(f"http://127.0.0.1:8000{endpoint}", timeout=(15, 60))
    print(f"\n== {endpoint} ==")
    print(r.status_code)
    print(json.dumps(r.json(), indent=2)[:4000])

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /api/v1/collections (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x76823ffe1f40>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [71]:
import os
import subprocess
import time
import requests
import json

os.chdir("/workspace/nz_legal_rag")

# Compile first
subprocess.run(
    ["python3", "-m", "py_compile", "core/rag_engine.py", "api/server.py"],
    check=False,
)

# Restart API
subprocess.run(["pkill", "-f", "uvicorn api.server:app"], check=False)

log = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(
    ["/workspace/nz_legal_rag/.venv/bin/python", "-m", "uvicorn", "api.server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("Started PID:", proc.pid)

# Wait for startup
time.sleep(6)

# Show log tail
print("\n=== /tmp/uvicorn.log ===")
try:
    with open("/tmp/uvicorn.log", "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
        print("".join(lines[-80:]))
except Exception as e:
    print("Could not read log:", e)

base = "http://127.0.0.1:8000"

# Probe root first
print("\n=== root probe ===")
try:
    r = requests.get(f"{base}/", timeout=(5, 15))
    print(r.status_code, r.text[:500])
except Exception as e:
    print("Root probe failed:", repr(e))

# Then hit actual endpoints
for endpoint in ["/api/v1/collections", "/health/deep"]:
    print(f"\n== {endpoint} ==")
    try:
        r = requests.get(f"{base}{endpoint}", timeout=(15, 60))
        print(r.status_code)
        try:
            print(json.dumps(r.json(), indent=2)[:4000])
        except Exception:
            print(r.text[:4000])
    except Exception as e:
        print("Request failed:", repr(e))

Started PID: 471976

=== /tmp/uvicorn.log ===
INFO:     Started server process [471976]
INFO:     Waiting for application startup.
Starting NZ Legal RAG API Server...
Environment CHROMA_DB_PATH=/workspace/chroma_db
Environment TENANT_DATA_PATH=./tenant_data
STEP 1: creating NZLegalRAG
Embedding model: nomic-embed-text:latest
Embedding model: nomic-embed-text:latest
STEP 1 OK: NZLegalRAG created
STEP 2: creating TenantManager
STEP 2 OK: TenantManager created
STEP 3: loading tenants
STEP 3 OK: tenants configured = 3
STEP 4: sweeping stale temp session directories
STEP 4 OK: Purged 0 temp dirs older than 12h
STEP 5: starting background temp sweeper
Application startup complete
TEMP SWEEP: Purged 0 temp dirs older than 12h
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
Shutting down...
INFO:     Application shutdown complete.
INFO:     Finished server process

In [72]:
from pathlib import Path
import re

path = Path("/workspace/nz_legal_rag/core/rag_engine.py")
text = path.read_text(encoding="utf-8")

targets = [
    "nz_legal_unified",
    "nzlii_criminal_cases",
    "legal_research",
    "user_uploads",
    "confidential",
]

original = text

for key in targets:
    patterns = [
        re.compile(rf'^\s*"{re.escape(key)}"\s*:\s*"[^"]*",\s*\n', re.M),
        re.compile(rf"^\s*'{re.escape(key)}'\s*:\s*'[^']*',\s*\n", re.M),
    ]
    for pattern in patterns:
        text = pattern.sub("", text)

if text == original:
    raise RuntimeError("No target collection entries were removed from core/rag_engine.py")

path.write_text(text, encoding="utf-8")

print("PATCHED:", path)
print("Removed collection entries:")
for key in targets:
    print(" -", key)

PATCHED: /workspace/nz_legal_rag/core/rag_engine.py
Removed collection entries:
 - nz_legal_unified
 - nzlii_criminal_cases
 - legal_research
 - user_uploads
 - confidential


In [73]:
import os
import subprocess
import time
import requests
import json

os.chdir("/workspace/nz_legal_rag")

subprocess.run(["python3", "-m", "py_compile", "core/rag_engine.py", "api/server.py"], check=False)
subprocess.run(["pkill", "-f", "uvicorn api.server:app"], check=False)

log = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(
    ["/workspace/nz_legal_rag/.venv/bin/python", "-m", "uvicorn", "api.server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("Started PID:", proc.pid)
time.sleep(6)

r = requests.get("http://127.0.0.1:8000/api/v1/collections", timeout=(15, 60))
print(r.status_code)
print(json.dumps(r.json(), indent=2))

Started PID: 472480
200
{
  "collections": [
    {
      "id": "nz_legislation",
      "description": "NZ Legislation",
      "document_count": 432854
    },
    {
      "id": "nz_case_law",
      "description": "NZ Case Law",
      "document_count": 9981
    },
    {
      "id": "nz_police_manual",
      "description": "NZ Police Manual",
      "document_count": 0
    },
    {
      "id": "confidential",
      "description": "Confidential Documents (Local)",
      "document_count": 0
    }
  ]
}


In [ ]:
import os
import math
import uuid
from collections import Counter
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHROMA_DB_PATH = "/workspace/chroma_db"
SOURCE_COLLECTION = "nz_legislation"
TARGET_COLLECTION = "nz_legislation_rechunked"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

src = client.get_collection(SOURCE_COLLECTION)

# Delete target first if it already exists
try:
    client.delete_collection(TARGET_COLLECTION)
    print(f"Deleted existing {TARGET_COLLECTION}")
except Exception:
    pass

dst = client.get_or_create_collection(TARGET_COLLECTION)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

total = src.count()
batch_size = 500
new_chunk_count = 0
source_doc_counter = Counter()

print(f"Source rows: {total}")

for offset in range(0, total, batch_size):
    batch = src.get(
        limit=batch_size,
        offset=offset,
        include=["documents", "metadatas"]
    )

    out_ids = []
    out_docs = []
    out_meta = []

    docs = batch.get("documents", []) or []
    metas = batch.get("metadatas", []) or []

    for i, doc in enumerate(docs):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        if not doc or not str(doc).strip():
            continue

        parent_id = (
            meta.get("doc_id")
            or meta.get("document_id")
            or meta.get("source")
            or f"row_{offset+i}"
        )

        source_doc_counter[parent_id] += 1

        chunks = splitter.split_text(doc)
        for j, chunk in enumerate(chunks):
            chunk_meta = dict(meta)
            chunk_meta["parent_doc_id"] = parent_id
            chunk_meta["rechunked_from"] = SOURCE_COLLECTION
            chunk_meta["chunk_index"] = j
            chunk_meta["chunk_count"] = len(chunks)

            out_ids.append(str(uuid.uuid4()))
            out_docs.append(chunk)
            out_meta.append(chunk_meta)

    if out_docs:
        dst.add(
            ids=out_ids,
            documents=out_docs,
            metadatas=out_meta,
        )
        new_chunk_count += len(out_docs)

    print(f"Processed {min(offset + batch_size, total)}/{total} source rows -> {new_chunk_count} new chunks")

print("\nDone.")
print("Original rows:", total)
print("New rechunked rows:", new_chunk_count)
print("Approx ratio:", round(new_chunk_count / max(total, 1), 2))
print("Target collection count:", dst.count())

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="/workspace/chroma_db")

print("All collections:")
for c in client.list_collections():
    name = getattr(c, "name", str(c))
    if "police" in name.lower() or "manual" in name.lower():
        try:
            count = client.get_collection(name).count()
        except Exception as e:
            count = f"ERROR: {e}"
        print(f" - {name}: {count}")

In [ ]:
import time
import uuid
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHROMA_DB_PATH = "/workspace/chroma_db"
SOURCE_COLLECTION = "nz_legislation"
TARGET_COLLECTION = "nz_legislation_rechunked"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
src = client.get_collection(SOURCE_COLLECTION)

try:
    client.delete_collection(TARGET_COLLECTION)
    print(f"Deleted existing {TARGET_COLLECTION}")
except Exception:
    pass

dst = client.get_or_create_collection(TARGET_COLLECTION)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

total = src.count()
batch_size = 50
new_chunk_count = 0

print(f"Source rows: {total}")
print(f"Target collection: {TARGET_COLLECTION}")
print("Starting...\n")

for offset in range(0, total, batch_size):
    t0 = time.time()

    batch = src.get(
        limit=batch_size,
        offset=offset,
        include=["documents", "metadatas"]
    )
    t1 = time.time()

    out_ids, out_docs, out_meta = [], [], []

    docs = batch.get("documents", []) or []
    metas = batch.get("metadatas", []) or []

    for i, doc in enumerate(docs):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        if not doc or not str(doc).strip():
            continue

        parent_id = (
            meta.get("doc_id")
            or meta.get("document_id")
            or meta.get("source")
            or f"row_{offset+i}"
        )

        chunks = splitter.split_text(doc)
        for j, chunk in enumerate(chunks):
            chunk_meta = dict(meta)
            chunk_meta["parent_doc_id"] = parent_id
            chunk_meta["rechunked_from"] = SOURCE_COLLECTION
            chunk_meta["chunk_index"] = j
            chunk_meta["chunk_count"] = len(chunks)

            out_ids.append(str(uuid.uuid4()))
            out_docs.append(chunk)
            out_meta.append(chunk_meta)

    t2 = time.time()

    print(
        f"[BATCH {offset}:{min(offset+batch_size, total)}] "
        f"fetched={len(docs)} split_into={len(out_docs)} "
        f"fetch={t1-t0:.2f}s split={t2-t1:.2f}s"
    )

    if out_docs:
        print(f"  -> adding {len(out_docs)} chunks to {TARGET_COLLECTION} ...", flush=True)
        t3 = time.time()
        dst.add(
            ids=out_ids,
            documents=out_docs,
            metadatas=out_meta,
        )
        t4 = time.time()
        new_chunk_count += len(out_docs)
        print(
            f"  -> add complete in {t4-t3:.2f}s | total_new_chunks={new_chunk_count}",
            flush=True
        )
    else:
        print("  -> no output chunks in this batch", flush=True)

print("\nDone.")
print("Original rows:", total)
print("New rechunked rows:", new_chunk_count)
print("Approx ratio:", round(new_chunk_count / max(total, 1), 2))
print("Target collection count:", dst.count())

In [ ]:
import time
import uuid
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHROMA_DB_PATH = "/workspace/chroma_db"
SOURCE_COLLECTION = "nz_legislation"
TARGET_COLLECTION = "nz_legislation_rechunked"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
src = client.get_collection(SOURCE_COLLECTION)

try:
    client.delete_collection(TARGET_COLLECTION)
    print(f"Deleted existing {TARGET_COLLECTION}")
except Exception:
    pass

dst = client.get_or_create_collection(TARGET_COLLECTION)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

total = src.count()
batch_size = 50
new_chunk_count = 0

print(f"Source rows: {total}")
print(f"Target collection: {TARGET_COLLECTION}")
print("Starting...\n")

for offset in range(0, total, batch_size):
    t0 = time.time()

    batch = src.get(
        limit=batch_size,
        offset=offset,
        include=["documents", "metadatas"]
    )
    t1 = time.time()

    out_ids, out_docs, out_meta = [], [], []

    docs = batch.get("documents", []) or []
    metas = batch.get("metadatas", []) or []

    for i, doc in enumerate(docs):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        if not doc or not str(doc).strip():
            continue

        parent_id = (
            meta.get("doc_id")
            or meta.get("document_id")
            or meta.get("source")
            or f"row_{offset+i}"
        )

        chunks = splitter.split_text(doc)
        for j, chunk in enumerate(chunks):
            chunk_meta = dict(meta)
            chunk_meta["parent_doc_id"] = parent_id
            chunk_meta["rechunked_from"] = SOURCE_COLLECTION
            chunk_meta["chunk_index"] = j
            chunk_meta["chunk_count"] = len(chunks)

            out_ids.append(str(uuid.uuid4()))
            out_docs.append(chunk)
            out_meta.append(chunk_meta)

    t2 = time.time()

    print(
        f"[BATCH {offset}:{min(offset+batch_size, total)}] "
        f"fetched={len(docs)} split_into={len(out_docs)} "
        f"fetch={t1-t0:.2f}s split={t2-t1:.2f}s"
    )

    if out_docs:
        print(f"  -> adding {len(out_docs)} chunks to {TARGET_COLLECTION} ...", flush=True)
        t3 = time.time()
        dst.add(
            ids=out_ids,
            documents=out_docs,
            metadatas=out_meta,
        )
        t4 = time.time()
        new_chunk_count += len(out_docs)
        print(
            f"  -> add complete in {t4-t3:.2f}s | total_new_chunks={new_chunk_count}",
            flush=True
        )
    else:
        print("  -> no output chunks in this batch", flush=True)

print("\nDone.")
print("Original rows:", total)
print("New rechunked rows:", new_chunk_count)
print("Approx ratio:", round(new_chunk_count / max(total, 1), 2))
print("Target collection count:", dst.count())

In [ ]:
r = requests.post(
    "http://127.0.0.1:8000/api/v1/disclosure/analyse",
    json=payload,
    timeout=(15, 900),
)

In [ ]:
from pathlib import Path

candidates = [
    Path("/workspace/nz_legal_rag/api/server.py"),
    Path("api/server.py"),
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise RuntimeError("Could not find api/server.py")

print("Using:", path)

lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

needles = ["disclosure", "analyse", "analyze", "BackgroundTasks", "job_id"]

matches = []
for i, line in enumerate(lines, start=1):
    low = line.lower()
    if any(n.lower() in low for n in needles):
        matches.append(i)

if not matches:
    raise RuntimeError("No disclosure/analyse-related lines found")

shown = set()
for ln in matches[:20]:
    start = max(1, ln - 6)
    end = min(len(lines), ln + 16)
    key = (start, end)
    if key in shown:
        continue
    shown.add(key)
    print("\n" + "=" * 90)
    print(f"Lines {start}-{end}")
    print("=" * 90)
    for j in range(start, end + 1):
        print(f"{j:4}: {lines[j-1]}")

In [ ]:
grep -nEi "disclosure|analyse|analy" api/server.py

In [ ]:
from uuid import uuid4
from datetime import datetime
from fastapi import BackgroundTasks

analysis_jobs = {}

def _run_analysis_job(job_id: str, request: AnalysisRequest, session_id: Optional[str], tenant_id: str):
    analysis_jobs[job_id] = {
        "status": "running",
        "created_at": analysis_jobs[job_id]["created_at"],
        "started_at": datetime.now().isoformat(),
    }
    try:
        if not rag_engine:
            raise RuntimeError("RAG engine not initialized")

        full_query = request.query
        if request.context:
            full_query = f"{request.query}\n\nContext: {request.context}"

        collections = _get_search_collections(session_id, None)
        analysis = rag_engine.legal_analysis(
            query=full_query,
            analysis_type=request.analysis_type,
            collections=collections,
            deep_analysis=request.deep_analysis,
        )

        if tenant_manager:
            tenant_manager.record_usage(tenant_id, query_count=1)

        analysis_jobs[job_id] = {
            "status": "completed",
            "created_at": analysis_jobs[job_id]["created_at"],
            "started_at": analysis_jobs[job_id].get("started_at"),
            "finished_at": datetime.now().isoformat(),
            "result": {
                "query": request.query,
                "answer": analysis.answer,
                "citations": analysis.citations,
                "confidence": analysis.confidence,
                "analysis_type": analysis.analysis_type,
                "sources": [
                    {
                        "title": s.metadata.get("title", "Unknown"),
                        "category": s.metadata.get("category", "Unknown"),
                        "relevance": round(s.relevance, 4),
                    }
                    for s in analysis.sources
                ],
                "timestamp": datetime.now().isoformat(),
                "executive_summary": getattr(analysis, "executive_summary", None),
                "audit_report": getattr(analysis, "audit_report", None),
                "strategic_notes": getattr(analysis, "strategic_notes", None),
                "confidence_breakdown": getattr(analysis, "confidence_breakdown", None),
                "metadata": getattr(analysis, "metadata", None),
                "agent_trace": getattr(analysis, "agent_trace", None),
            },
        }
    except Exception as e:
        analysis_jobs[job_id] = {
            "status": "failed",
            "created_at": analysis_jobs[job_id]["created_at"],
            "started_at": analysis_jobs[job_id].get("started_at"),
            "finished_at": datetime.now().isoformat(),
            "error": str(e),
        }

@app.post("/api/v1/analyze", status_code=202)
def analyze(
    request: AnalysisRequest,
    background_tasks: BackgroundTasks,
    session_id: Optional[str] = Header(None, alias="X-Session-ID"),
    tenant=Depends(get_current_tenant),
):
    if not rag_engine:
        raise HTTPException(status_code=500, detail="RAG engine not initialized")

    check_quota(tenant, "query")

    job_id = str(uuid4())
    analysis_jobs[job_id] = {
        "status": "queued",
        "created_at": datetime.now().isoformat(),
    }

    background_tasks.add_task(
        _run_analysis_job,
        job_id,
        request,
        session_id,
        tenant.tenant_id,
    )

    return {
        "job_id": job_id,
        "status": "queued",
        "timestamp": datetime.now().isoformat(),
    }

@app.get("/api/v1/analyze/{job_id}")
def get_analysis_job(job_id: str, tenant=Depends(get_current_tenant)):
    job = analysis_jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Analysis job not found")
    return job